# KLTN attack v3.3 — Auto-captioning + Dynamic Concept Attention (Kaggle T4 ×2)

**New in v3.3:** source/target captions are **generated automatically** from the **actual preprocessed attack images**. No embedded 100-caption table. By default, `CAPTION_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'`: the same NF4/FP16 model already used for attention on GPU 1, reused without loading another copy. Set `CAPTION_MODEL_ID` to another compatible Hugging Face image-text generation VLM to use it instead. Different caption models run a separate cached caption prepass on GPU 1 and are unloaded before the Qwen attention backend is loaded (avoid simultaneous large VLMs on a T4).

**Dataset:** use `PAIR_MANIFEST_PATH = None` for a **one-to-one match by filename stem** (e.g. `bigscale/nips17/0.png` ↔ `target_images/1/0.jpg`), or provide a JSONL pair manifest for custom mappings (`original_path`, `target_path`, optional `sample_id`). Ambiguous stems are rejected rather than silently matched by row order. `TARGET_SUBDIR='1'` preserves the reference target class; set `None` when target stems are globally unique. `NUM_SAMPLES=None` processes **all matching pairs**, without the 100-image restriction; use e.g. 1 for a smoke test. `SAMPLE_START` supports shards.

**Running:** Kaggle → Internet On → Accelerator T4 ×2 → Add dataset input → set `DATASET_ROOT` → Run All in a fresh session. GPU 0 holds CLIP FOA B16/B32/Laion, GPU 1 Qwen3-VL. `ATTENTION_MODE='dynamic'` recomputes saliency on current adversarial PNG checkpoints; text concepts are *frozen for each sample* (not re-captioned on every update). FOA global/local OT, ensemble, full/crop updates, and epsilon projection are unchanged from v3.2.

**Caption audit/recovery:** the pair gets `captions/<sample_id>.json` (source/target text, SHA256 of preprocessed image PNG bytes, HF checkpoint revision, prompt/generation settings). Cache files are written atomically; `captions.jsonl` is exported from cache at the end, including for partial runs. Interrupted attacks reuse only captions whose preprocessed image hashes and caption-model/config match; a mismatch is a hard error, not a fallback to hard-coded captions. `GUIDE_CAPTION_ROLE='source'` by default; change to `'target'` or use explicit overrides. Adversarial captions are *not* used as pre-attack guides because they do not exist before generation.

**Artifacts:** per-sample attack source/target/adversarial images, attention NPZ/JSON, checkpoint PNGs, timing/steps, `manifest.jsonl`, `run_config.json`, `model_metadata.json`, `captions.jsonl`, archive ZIP. Can disable expensive plotting by setting `PLOT_FIRST_N_SAMPLES=0`. Resume with `RESUME_RUN_DIR=Path('/kaggle/working/.../kltn_attack_v3_3_dynamic_...')` using the **same v3.3 code and configs**. v3.2 runs cannot resume under v3.3 because data-selection/identity changed.

**Attention fidelity:** SDPA prefix, selected Q/K rows at layer 29, causal softmax over all keys, early exit, fresh per-image visual cache. Concept audit/hash and checkpoint behavior are maintained. Source/reference: [FOA-Attack](https://github.com/jiaxiaojunQAQ/FOA-Attack), [Qwen3-VL model](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct), [Transformers image-text generation](https://huggingface.co/docs/transformers/main/tasks/image_text_to_text). This edited notebook is **statically tested**; actual model inference and T4×2 performance must be verified on Kaggle.


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
OUTPUT_ROOT = WORK_ROOT / 'kltn_attack_v3_3_outputs'
REPO_DIR = WORK_ROOT / '_foa_attention_reference'
HF_HOME = Path('/kaggle/temp/foa_attention_hf_cache')
FOA_REPO_URL = 'https://github.com/jiaxiaojunQAQ/FOA-Attack.git'
FOA_REVISION = '6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b'
KMEANS_REVISION = 'f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805'

NUM_SAMPLES = None            # None = ALL pairs; set 1 for smoke test, or any positive N.
STEPS = 300
PAPER_STEPS = 300
PAPER_EPSILON = 16.0        # attack tensor trong [0,255]
PAPER_ALPHA = 1.0
PAPER_INPUT_RES = 224
PAPER_CROP_SCALE = (0.5, 0.9)
PAPER_BACKBONES = ('B16', 'B32', 'Laion')
PAPER_CLUSTER_NUMBER = 10
PAPER_SEED = 2023
ATTACK_DEVICE = 'cuda:0'
ATTENTION_DEVICE = 'cuda:1'
ATTENTION_MODE = 'dynamic'    # fixed | dynamic | baseline
NUM_PHASES = 10
HOTSPOT_TOP_K = 3           # khác với khoảng checkpoint k
HOTSPOT_IOU = 0.3
CHECKPOINT_INTERVAL = max(1, STEPS // 10)  # derived, không chỉnh riêng

# Dataset pairing (no fixed 100-caption table): exact filename-stem matching or JSONL mapping.
PAIR_MANIFEST_PATH = None       # JSONL rows: original_path, target_path, optional sample_id
TARGET_SUBDIR = '1'            # target_images/1; None for unique stems under all target_images
PAIRING_STRICT = False         # True: fail if ANY source has no corresponding target

# Captions generated on the 224x224 RGB FOA inputs, once per pair, then cached on disk.
CAPTION_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'  # editable: Hugging Face VLM
CAPTION_MODEL_REVISION = 'main'
CAPTION_DEVICE = 'cuda:1'
CAPTION_PROMPT = ('Describe the visible content of this image in one precise English sentence '
                  'of at most 25 words. Do not add introductions or unsupported details.')
CAPTION_MAX_NEW_TOKENS = 64
CAPTION_QUANTIZATION = 'nf4'    # used for a different caption model; shared Qwen uses attention NF4
GUIDE_CAPTION_ROLE = 'source'  # 'source' or 'target'; not 'adversarial' before attack
GUIDE_TEXT_OVERRIDE = None  # vd 'a panda bear'; áp dụng chung, hoặc dùng dict theo sample_id.
GUIDE_OVERRIDES = {}        # vd {'00000': 'a panda bear'}; ưu tiên hơn override chung.
EXTRA_CONCEPTS = []         # vd ['a dog', 'a toothbrush']; thêm vào source/target/guide.
EXTRA_CONCEPTS_BY_SAMPLE = {}

ATTENTION_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
ATTENTION_REVISION = 'main' # commit đã resolve được ghi vào model_metadata.json
ATTENTION_LAYER = 29        # zero-based, theo config SAGA
ATTENTION_INPUT_RES = 448
MAX_CONCEPT_TOKENS = 128
CONTENT_WORDS_ONLY = True
ATTENTION_PROMPT = 'Describe this image, no longer than 25 words.'
PRUNE_TEXT_BRANCH = True    # loss FOA không dùng text branch CLIP
RUN_ATTACK = True
INSTALL_DEPENDENCIES = True
RUN_PARITY_CHECK = True
LIVE_PREVIEW = False

if CAPTION_DEVICE != ATTENTION_DEVICE:
    raise ValueError('Caption model must use the attention GPU (no simultaneous second large model).')
if CAPTION_MAX_NEW_TOKENS < 1 or CAPTION_QUANTIZATION not in {'nf4', 'fp16'}:
    raise ValueError('Invalid caption generation settings.')
if SAMPLE_START < 0 or not isinstance(SAMPLE_START, int):
    raise ValueError('SAMPLE_START must be a nonnegative integer.')
if ATTENTION_MODE not in {'fixed', 'dynamic', 'baseline'}:
    raise ValueError('ATTENTION_MODE must be fixed, dynamic, or baseline.')
if type(STEPS) is not int or STEPS < 1:
    raise ValueError('STEPS must be a positive integer.')
if NUM_SAMPLES is not None and (type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1):
    raise ValueError('NUM_SAMPLES must be positive or None.')
if GUIDE_CAPTION_ROLE not in {'source', 'target'}:
    raise ValueError('Invalid GUIDE_CAPTION_ROLE.')

# Throughput knobs. Fixed for a run; saved with the fingerprints.
FOA_AUTOCAST = True              # False: FP32 CLIP ablation; OT/K-means always FP32
GRADIENT_SCALE = 1024.0 if FOA_AUTOCAST else 1.0
OBSERVE_ALL_CONCEPTS_EVERY = 5    # checkpoint ordinals; 0=endpoints only; 1=every checkpoint
ASYNC_OBSERVATION = True         # guide remains synchronous/fresh; only auxiliary maps overlap
PLOT_FIRST_N_SAMPLES = None         # None: every sample; 0: maps only
SHOW_FINAL_PLOTS = False
PROGRESS_INTERVAL = 30
RESUME_RUN_DIR = None            # writable existing run directory, same configuration
SAMPLE_START = 0                 # offset in sorted source/target matches; supports shards
CONTINUE_ON_ERROR = True
ZIP_COMPRESSION_LEVEL = 1

if type(OBSERVE_ALL_CONCEPTS_EVERY) is not int or OBSERVE_ALL_CONCEPTS_EVERY < 0:
    raise ValueError('OBSERVE_ALL_CONCEPTS_EVERY must be a nonnegative integer.')
if not 0 < GRADIENT_SCALE < float('inf') or PROGRESS_INTERVAL < 1:
    raise ValueError('Invalid gradient scale/progress interval.')
if PLOT_FIRST_N_SAMPLES is not None and (type(PLOT_FIRST_N_SAMPLES) is not int or PLOT_FIRST_N_SAMPLES < 0):
    raise ValueError('PLOT_FIRST_N_SAMPLES must be nonnegative or None.')
if type(SAMPLE_START) is not int or SAMPLE_START < 0:
    raise ValueError('SAMPLE_START must be a nonnegative integer.')

if not (0 <= PAPER_EPSILON <= 255) or not (0 < PAPER_ALPHA < float('inf')):
    raise ValueError('epsilon must be in [0,255] and alpha must be finite and positive.')
if type(PROGRESS_INTERVAL) is not int or not 0 <= ZIP_COMPRESSION_LEVEL <= 9:
    raise ValueError('Invalid progress interval or ZIP compression level.')
if not PAPER_BACKBONES or any(name not in {'B16', 'B32', 'Laion'} for name in PAPER_BACKBONES):
    raise ValueError('Select valid FOA backbones.')
if type(PAPER_CLUSTER_NUMBER) is not int or not 1 < PAPER_CLUSTER_NUMBER <= 49:
    raise ValueError('K-means needs 2..49 clusters for the B32 patch grid.')


In [ ]:
# Một môi trường package cho cả CLIP/FOA và Qwen. Không cài lại Torch/Torchvision.
import os, sys, subprocess, json, hashlib, time, random, shutil, ast, traceback, inspect
from datetime import datetime, timezone

PACKAGES = [
    'transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface-hub==0.34.4',
    'accelerate==1.11.0', 'bitsandbytes==0.48.1', 'ftfy==6.3.1',
    'tqdm==4.67.1', 'loguru==0.7.3',
]
SETUP_DIR = WORK_ROOT / '_attention_v3_setup'

def quiet_command(command, name):
    SETUP_DIR.mkdir(parents=True, exist_ok=True)
    log_path = SETUP_DIR / name
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'Setup failed: {log_path}')

if INSTALL_DEPENDENCIES:
    if any(name in sys.modules for name in ('transformers', 'tokenizers', 'kmeans_pytorch', 'surrogates')):
        raise RuntimeError('Restart session before installing pinned packages, or set INSTALL_DEPENDENCIES=False.')
    quiet_command([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], 'pip.log')
    quiet_command([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
                   f'kmeans-pytorch @ git+https://github.com/subhadarship/kmeans_pytorch.git@{KMEANS_REVISION}'], 'kmeans.log')
os.environ.update({'HF_HOME': str(HF_HOME), 'TOKENIZERS_PARALLELISM': 'false',
                   'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'WANDB_MODE': 'disabled'})
HF_HOME.mkdir(parents=True, exist_ok=True)
# Dedicated checkout: không xóa/ghi đè checkout v1.2 hoặc v2 của bạn.
if not REPO_DIR.exists():
    quiet_command(['git', 'clone', FOA_REPO_URL, str(REPO_DIR)], 'git_clone.log')
if not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'Expected a dedicated Git checkout: {REPO_DIR}')
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.rstrip('/') != FOA_REPO_URL.rstrip('/'):
    raise RuntimeError('FOA checkout origin differs from configured source.')
dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
if dirty.strip():
    raise RuntimeError('Dedicated FOA checkout has local changes; choose a new REPO_DIR.')
quiet_command(['git', '-C', str(REPO_DIR), 'checkout', '--detach', FOA_REVISION], 'git_checkout.log')
FOA_SOURCE_REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if FOA_SOURCE_REVISION != FOA_REVISION:
    raise RuntimeError('FOA revision mismatch.')
sys.path.insert(0, str(REPO_DIR))
print('Setup ready. FOA:', FOA_SOURCE_REVISION)


In [ ]:
import contextlib, io, csv, gc, math, string, textwrap, threading, zipfile
from concurrent.futures import ThreadPoolExecutor, Future
from types import MappingProxyType
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Any, Optional, Sequence
from importlib import metadata
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF, InterpolationMode
from torchvision.utils import save_image
from kmeans_pytorch import kmeans
from surrogates import ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor, EnsembleFeatureExtractor_ot
from transformers import AutoConfig, AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from IPython import get_ipython
from IPython.display import display, FileLink
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from loguru import logger
logger.remove()
logger.add(sys.stderr, level='WARNING')
if metadata.version('transformers') != '4.57.1':
    raise RuntimeError('This Qwen Q/K capture targets transformers 4.57.1; install pins and restart session.')

CODE_SNAPSHOTS = {}
def remember_code(name):
    shell = get_ipython()
    if shell is not None:
        CODE_SNAPSHOTS[name] = shell.history_manager.input_hist_raw[-1]

def utc_stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

@contextlib.contextmanager
def preserve_rng():
    python_state, numpy_state, cpu_state = random.getstate(), np.random.get_state(), torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        torch.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

def set_foa_seed(seed=PAPER_SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def validate_devices():
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError('Attack requires two GPUs: enable Kaggle T4 x2; model loading has not started.')
    devices = [torch.device(ATTACK_DEVICE), torch.device(ATTENTION_DEVICE)]
    if any(d.type != 'cuda' or d.index is None or d.index >= torch.cuda.device_count() for d in devices):
        raise ValueError('Configure two distinct available explicit CUDA devices.')
    if devices[0] == devices[1]:
        raise ValueError('FOA and Qwen must use different GPUs.')
    for d in devices:
        print(d, torch.cuda.get_device_name(d))

if 'iter_limit' not in inspect.signature(kmeans).parameters:
    raise RuntimeError('K-means lacks iter_limit; restart after the pinned install.')
direct_url = json.loads(metadata.distribution('kmeans-pytorch').read_text('direct_url.json') or '{}')
if direct_url.get('vcs_info', {}).get('commit_id') != KMEANS_REVISION:
    raise RuntimeError('K-means is not the required pinned Git revision.')
RUN_DIR = None


def concept_sha256(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Concept must be nonempty text.')
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def recover_completed_records(run_dir, selected_ids):
    """Recover complete sample commits; tolerate only a torn final manifest line."""
    run_dir = Path(run_dir)
    manifest_path = run_dir / 'manifest.jsonl'
    manifest_records = {}
    if manifest_path.is_file():
        raw = manifest_path.read_bytes()
        lines = raw.splitlines(keepends=True)
        for index, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except (ValueError, UnicodeDecodeError):
                if index != len(lines) - 1 or line.endswith(b'\n'):
                    raise ValueError('Manifest has a malformed committed/middle line; refusing to discard it.')
                backup = run_dir / f'manifest_torn_{utc_stamp()}.jsonl'
                backup.write_bytes(raw)
                break
            sample_id = record['sample_id']
            if sample_id in manifest_records:
                raise ValueError(f'Duplicate manifest sample: {sample_id}')
            manifest_records[sample_id] = record
    records = {}
    samples_dir = run_dir / 'samples'
    for sample_dir in sorted(samples_dir.iterdir()) if samples_dir.is_dir() else []:
        status_path = sample_dir / 'status.json'
        if not sample_dir.is_dir() or not status_path.is_file():
            continue
        sample_status = json.loads(status_path.read_text(encoding='utf-8'))
        if sample_status.get('status') != 'complete':
            continue
        if sample_dir.name not in selected_ids:
            raise ValueError(f'Completed sample is outside selected data: {sample_dir.name}')
        record_path = sample_dir / 'manifest_record.json'
        record = (json.loads(record_path.read_text(encoding='utf-8')) if record_path.is_file()
                  else manifest_records.get(sample_dir.name))
        if record is None or record.get('sample_id') != sample_dir.name:
            raise ValueError(f'Completed sample has no valid manifest record: {sample_dir.name}')
        records[sample_dir.name] = record
    return records

def attention_token_fingerprint(metadata):
    keys = ('token_ids', 'selected_offsets', 'concept_query_rows', 'vision_positions', 'grid_shape', 'method', 'layer')
    if any(key not in metadata for key in keys):
        raise ValueError('Attention metadata lacks token-alignment proof.')
    return hashlib.sha256(json.dumps({key: metadata[key] for key in keys}, sort_keys=True).encode()).hexdigest()


def validate_completed_artifacts(run_dir, record, steps):
    run_dir = Path(run_dir)
    sample_id = record['sample_id']
    sample_dir = run_dir / 'samples' / sample_id
    guide_sha = concept_sha256(record['guide_concept'])
    token_signature = None
    for role in ('source', 'target', 'adversarial'):
        image_path = (run_dir / record[role]).resolve()
        if not image_path.is_relative_to(run_dir.resolve()):
            raise ValueError('Manifest image escapes the run directory.')
        if sha256_file(image_path) != record['image_sha256'][role]:
            raise ValueError(f'Resume image checksum changed: {sample_id}/{role}')
    for step in checkpoint_steps(steps):
        image_path = sample_dir / 'checkpoints' / f'step_{step:05d}.png'
        map_dir = sample_dir / 'attention' / f'step_{step:05d}'
        metadata = json.loads((map_dir / 'guide.json').read_text(encoding='utf-8'))
        if metadata.get('checkpoint_sha256') != sha256_file(image_path):
            raise ValueError(f'Resume checkpoint checksum changed: {sample_id}/{step}')
        if metadata.get('concept_sha256') != guide_sha or metadata.get('concept') != record['guide_concept']:
            raise ValueError(f'Resume guide concept changed: {sample_id}/{step}')
        signature = metadata.get('token_signature_sha256')
        if (not isinstance(signature, str) or signature != attention_token_fingerprint(metadata)
                or (token_signature is not None and signature != token_signature)):
            raise ValueError(f'Resume guide tokens changed: {sample_id}/{step}')
        token_signature = signature
        if metadata.get('map_sha256') != sha256_file(map_dir / 'guide.npz'):
            raise ValueError(f'Resume attention map checksum changed: {sample_id}/{step}')
        with np.load(map_dir / 'guide.npz', allow_pickle=False) as saved:
            probability, minmax = saved['probability'], saved['minmax']
            if (probability.ndim != 2 or minmax.shape != probability.shape or not np.isfinite(probability).all()
                    or probability.min() < 0 or not np.isclose(probability.sum(), 1, atol=1e-5)
                    or not np.isfinite(minmax).all() or minmax.min() < 0 or minmax.max() > 1 + 1e-5):
                raise ValueError(f'Resume attention map is invalid: {sample_id}/{step}')
    if sha256_file(sample_dir / 'checkpoints' / f'step_{steps:05d}.png') != record['image_sha256']['adversarial']:
        raise ValueError(f'Final image differs from its final checkpoint: {sample_id}')

remember_code('imports')


In [ ]:
# v3.3: Captions are generated at runtime from source/target images. No frozen caption table.
REFERENCE_PROVENANCE = {
    'base_notebook': 'kltn-attack-v3-2.ipynb',
    'base_notebook_sha256': 'f0f47d290817e0648384d1fdbdec58f8d9fc8f9551e517faee0936afd8de5d51',
    'caption_policy': 'generated-from-preprocessed-attack-inputs',
}
V33_PARENT_NOTEBOOK_SHA256 = 'f0f47d290817e0648384d1fdbdec58f8d9fc8f9551e517faee0936afd8de5d51'
remember_code('reference_provenance')


In [ ]:
# Cell 5 — Dataset discovery and preprocessing (editable research boundary).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(path: Path) -> int:
    """Return recursive number of supported image files below ``path``."""
    return sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))

def find_imagefolder(root: Path, expected_name: str) -> Path:
    """Find the largest ImageFolder named ``expected_name`` in a Kaggle dataset."""
    candidates = sorted((p for p in root.rglob(expected_name) if p.is_dir()), key=lambda p: -count_images(p))
    if not candidates: raise FileNotFoundError(f'Cannot find {expected_name} below {root}')
    return candidates[0]

def pil_to_foa_tensor(image: Image.Image) -> Tensor:
    """Convert PIL RGB image to the original FOA tensor in pixel range [0, 255]."""
    array = np.array(image, np.uint8, copy=True)
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(torch.get_default_dtype())


# Preprocessing
def preprocess_image(input_res: int = PAPER_INPUT_RES) -> transforms.Compose:
    """Build the exact Resize → CenterCrop → RGB → pixel-tensor preprocessing of FOA."""
    return transforms.Compose([
        transforms.Resize(input_res, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(input_res), transforms.Lambda(lambda x: x.convert('RGB')),
        transforms.Lambda(pil_to_foa_tensor),
    ])

class ImageFolderWithPaths(datasets.ImageFolder):
    """ImageFolder that additionally returns the source image path for manifest export."""
    def __getitem__(self, index: int):
        """Return transformed image, class label, and original filesystem path."""
        image, label = super().__getitem__(index)
        return image, label, self.samples[index][0]


def relative_dataset_path(path, anchor):
    parts = str(path).replace('\\', '/').split('/')
    positions = [i for i, part in enumerate(parts) if part == anchor]
    if len(positions) != 1 or positions[0] == len(parts) - 1:
        raise ValueError(f'Cannot identify unique dataset anchor {anchor}: {path}')
    relative = '/'.join(parts[positions[0] + 1:])
    if any(part in {'', '.', '..'} for part in relative.split('/')):
        raise ValueError('Unsafe relative dataset path.')
    return relative

def pair_key(source_path, target_path):
    return (relative_dataset_path(source_path, 'bigscale'), relative_dataset_path(target_path, 'target_images'))

def tensor_png_bytes(pixel_tensor):
    # Exactly torchvision save_image's rounding, with PNG lossless serialization.
    image = pixel_tensor.detach().cpu()
    if image.ndim == 4:
        if image.shape[0] != 1:
            raise ValueError('Only one image per checkpoint.')
        image = image[0]
    array = image.add(0.5).clamp(0, 255).permute(1, 2, 0).to(torch.uint8).numpy()
    buffer = io.BytesIO()
    Image.fromarray(array).save(buffer, format='PNG')
    return buffer.getvalue()

def png_digest_after_preprocess(path, transform):
    with Image.open(path) as image:
        pixels = transform(image)
    return hashlib.sha256(tensor_png_bytes(pixels)).hexdigest()

def _supported_image_paths(directory):
    return sorted((p for p in Path(directory).rglob('*')
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS), key=lambda p: p.as_posix())


def _manifest_image_path(value, expected_root):
    path = Path(value)
    if not path.is_absolute():
        # Manifest paths are dataset-root-relative (e.g. images/bigscale/nips17/0.png).
        path = DATASET_ROOT / path
    path = path.resolve()
    if not path.is_file() or not path.is_relative_to(expected_root.resolve()):
        raise ValueError(f'Manifest path is missing/outside its expected image root: {value}')
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        raise ValueError(f'Not an image: {path}')
    return path


def discover_pairs():
    """Return stable source-indexed pairs. No caption or fixed-table dependencies."""
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f'Add dataset input and correct DATASET_ROOT: {DATASET_ROOT}')
    source_root = find_imagefolder(DATASET_ROOT, 'bigscale')
    target_root = find_imagefolder(DATASET_ROOT, 'target_images')
    sources = _supported_image_paths(source_root)
    if not sources:
        raise ValueError(f'No source images under {source_root}')
    source_index = {p.resolve(): i for i, p in enumerate(sources)}
    pairs = []
    if PAIR_MANIFEST_PATH is not None:
        path = Path(PAIR_MANIFEST_PATH)
        with path.open('r', encoding='utf-8') as stream:
            for line_no, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                entry = json.loads(line)
                source_path = _manifest_image_path(entry.get('original_path', entry.get('source_path')), source_root)
                target_path = _manifest_image_path(entry['target_path'], target_root)
                sample_id = str(entry.get('sample_id', f'{source_index[source_path]:05d}'))
                pairs.append({'index': source_index[source_path], 'sample_id': sample_id,
                              'source_path': str(source_path), 'target_path': str(target_path)})
    else:
        target_folder = target_root if TARGET_SUBDIR is None else target_root / TARGET_SUBDIR
        if not target_folder.is_dir():
            raise FileNotFoundError(f'TARGET_SUBDIR does not exist: {target_folder}')
        targets_by_stem = {}
        for target in _supported_image_paths(target_folder):
            targets_by_stem.setdefault(target.stem, []).append(target)
        missing = []
        source_stems = set()
        for index, source in enumerate(sources):
            if source.stem in source_stems:
                raise ValueError(f'Duplicate source stem {source.stem!r}. Supply PAIR_MANIFEST_PATH.')
            source_stems.add(source.stem)
            candidates = targets_by_stem.get(source.stem, [])
            if len(candidates) > 1:
                raise ValueError(f'Ambiguous targets for {source}. Supply PAIR_MANIFEST_PATH.')
            if not candidates:
                missing.append(str(source))
                continue
            target = candidates[0]
            pairs.append({'index': index, 'sample_id': f'{index:05d}',
                          'source_path': str(source.resolve()), 'target_path': str(target.resolve())})
        if missing:
            print(f'No matching target for {len(missing)} of {len(sources)} sources. First: {missing[0]}')
            if PAIRING_STRICT:
                raise ValueError('PAIRING_STRICT=True: every source requires a matched target.')
    ids, source_target_keys = set(), set()
    for pair in pairs:
        sid = pair['sample_id']
        if not sid or not all(ch.isalnum() or ch in '_-' for ch in sid):
            raise ValueError(f'Unsafe sample ID: {sid!r}')
        pair['pair_key'] = pair_key(pair['source_path'], pair['target_path'])
        if sid in ids or pair['pair_key'] in source_target_keys:
            raise ValueError(f'Duplicate sample_id or image pair: {sid}')
        ids.add(sid); source_target_keys.add(pair['pair_key'])
    pairs.sort(key=lambda p: (p['index'], p['sample_id']))
    return pairs


def resolve_concepts(row):
    sample_id = row['sample_id']
    guide = GUIDE_OVERRIDES.get(sample_id, GUIDE_TEXT_OVERRIDE)
    if guide is None:
        guide = row['captions'][GUIDE_CAPTION_ROLE]
    candidates = {'source': row['captions']['source'], 'target': row['captions']['target'], 'guide': guide}
    extra = list(EXTRA_CONCEPTS) + list(EXTRA_CONCEPTS_BY_SAMPLE.get(sample_id, []))
    candidates.update({f'extra_{i:02d}': phrase for i, phrase in enumerate(extra)})
    for label, phrase in candidates.items():
        if not isinstance(phrase, str) or not phrase.strip():
            raise ValueError(f'Empty/invalid concept {label}: {sample_id}')
    return guide, candidates


def prepare_pairs():
    matched = discover_pairs()
    # Sample IDs for overrides must exist in the full matched set (not only this shard).
    all_ids = {row['sample_id'] for row in matched}
    for setting, overrides in [('GUIDE_OVERRIDES', GUIDE_OVERRIDES),
                               ('EXTRA_CONCEPTS_BY_SAMPLE', EXTRA_CONCEPTS_BY_SAMPLE)]:
        unknown = set(overrides) - all_ids
        if unknown:
            raise ValueError(f'{setting} contains unknown sample IDs: {sorted(unknown)[:10]}')
    matched = matched[SAMPLE_START:]
    if not matched:
        raise ValueError('No source/target image pairs match the current shard.')
    selected = matched if NUM_SAMPLES is None else matched[:NUM_SAMPLES]
    print(f'Matched {len(matched)} pairs after SAMPLE_START; selected {len(selected)}. '
          'Captions will be generated from attack inputs (not embedded).')
    origin = {'caption_source': 'generated', 'caption_model_id': CAPTION_MODEL_ID,
              'pair_manifest_path': str(PAIR_MANIFEST_PATH) if PAIR_MANIFEST_PATH else None,
              'pair_manifest_sha256': sha256_file(PAIR_MANIFEST_PATH) if PAIR_MANIFEST_PATH else None,
              'pairing_mode': 'manifest' if PAIR_MANIFEST_PATH else 'matching_stem',
              'target_subdir': TARGET_SUBDIR, 'matched_pairs_after_offset': len(matched)}
    return selected, origin

remember_code('data')


In [ ]:
# Cell 5b — On-demand and resumable VLM-generated captions (no fixed caption records).
# Exactly the image tensor written to the attack PNG is given to the caption model.

def caption_generation_config():
    return {'model_id': CAPTION_MODEL_ID, 'requested_revision': CAPTION_MODEL_REVISION,
            'device': CAPTION_DEVICE, 'prompt': CAPTION_PROMPT,
            'max_new_tokens': CAPTION_MAX_NEW_TOKENS, 'do_sample': False,
            'quantization': CAPTION_QUANTIZATION}


def reuse_attention_model_for_captioning():
    return (CAPTION_MODEL_ID == ATTENTION_MODEL_ID and
            CAPTION_MODEL_REVISION == ATTENTION_REVISION and
            CAPTION_QUANTIZATION == 'nf4' and CAPTION_DEVICE == ATTENTION_DEVICE)


class VLMCaptioner:
    """Use existing Qwen3-VL attention weights, or temporarily load another HF VLM."""
    def __init__(self, attention_backend=None, resolved_revision=None):
        self.reused = attention_backend is not None
        self.device = torch.device(CAPTION_DEVICE)
        if self.reused:
            if not reuse_attention_model_for_captioning():
                raise ValueError('Attention backend is only reusable with identical model/revision/NF4/device.')
            self.model = attention_backend.model
            self.processor = attention_backend.processor
            self.lock = attention_backend._lock
            revision = attention_backend.metadata['resolved_revision']
        else:
            requested = resolved_revision or CAPTION_MODEL_REVISION
            config = AutoConfig.from_pretrained(CAPTION_MODEL_ID, revision=requested)
            revision = getattr(config, '_commit_hash', None) or requested
            self.processor = AutoProcessor.from_pretrained(CAPTION_MODEL_ID, revision=revision)
            quant = None
            if CAPTION_QUANTIZATION == 'nf4':
                quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
                    llm_int8_skip_modules=['visual'])
            self.model = AutoModelForImageTextToText.from_pretrained(
                CAPTION_MODEL_ID, revision=revision, torch_dtype=torch.float16,
                quantization_config=quant, device_map={'': self.device.index},
                attn_implementation='sdpa', low_cpu_mem_usage=True,
                trust_remote_code=False).eval()
            self.model.requires_grad_(False)
            self.lock = threading.RLock()
            revision = getattr(self.model.config, '_commit_hash', None) or revision
        self.metadata = {**caption_generation_config(), 'resolved_revision': revision,
                         'reused_attention_model': self.reused}

    @torch.inference_mode()
    def caption(self, image):
        with self.lock, torch.cuda.device(self.device):
            if self.model.training:
                raise RuntimeError('Caption model must be in eval mode.')
            # Processor-chat-template and image processing are standard HF image-text interfaces.
            messages = [{'role': 'user', 'content': [
                {'type': 'image'}, {'type': 'text', 'text': CAPTION_PROMPT}]}]
            prompt = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
            inputs = self.processor(text=[prompt], images=[image.convert('RGB')],
                                    return_tensors='pt')
            inputs = {key: val.to(self.device) if isinstance(val, torch.Tensor) else val
                      for key, val in inputs.items()}
            input_length = inputs['input_ids'].shape[-1]
            # The attention early-exit hook is installed only during attention extraction.
            if self.reused and getattr(self.model, 'model', None) is not None:
                self.model.model.rope_deltas = None
            generated = self.model.generate(**inputs, max_new_tokens=CAPTION_MAX_NEW_TOKENS,
                                             do_sample=False, num_beams=1, use_cache=True)
            tail = generated[0, input_length:]
            text = self.processor.decode(tail, skip_special_tokens=True,
                                         clean_up_tokenization_spaces=True)
            text = ' '.join(text.strip().split())
            if not text:
                raise RuntimeError('Caption model returned an empty caption; not using any fallback.')
            return text

    def close(self):
        if not self.reused:
            del self.model, self.processor
            gc.collect()
            if torch.cuda.is_available():
                with torch.cuda.device(self.device):
                    torch.cuda.empty_cache()


def _caption_images_from_tensors(source, target):
    source_bytes = tensor_png_bytes(source)
    target_bytes = tensor_png_bytes(target)
    source_image = Image.open(io.BytesIO(source_bytes)).convert('RGB')
    target_image = Image.open(io.BytesIO(target_bytes)).convert('RGB')
    return (source_bytes, target_bytes), (source_image, target_image)


def get_or_generate_caption_record(run_dir, pair, captioner, image_bytes, images):
    """Write per-pair cache atomically. Verify both images before reusing cached captions."""
    path = Path(run_dir) / 'captions' / f'{pair["sample_id"]}.json'
    expected_hashes = {'source': hashlib.sha256(image_bytes[0]).hexdigest(),
                       'target': hashlib.sha256(image_bytes[1]).hexdigest()}
    expected_config = caption_generation_config()
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        if (saved.get('sample_id') != pair['sample_id'] or
            saved.get('original_path') != pair['source_path'] or
            saved.get('target_path') != pair['target_path'] or
            saved.get('image_sha256') != expected_hashes or
            saved.get('caption_config') != expected_config):
            raise ValueError(f'Cached caption/input/config mismatch: {path}. '
                             'Use the original images/config or start a NEW run.')
        recorded_revision = saved.get('caption_model', {}).get('resolved_revision')
        if captioner is not None and recorded_revision != captioner.metadata['resolved_revision']:
            raise ValueError(f'Caption model revision changed: {path}')
        for role in ('source', 'target'):
            if not isinstance(saved.get('captions', {}).get(role), str) or not saved['captions'][role].strip():
                raise ValueError(f'Invalid cached {role} caption: {path}')
        return saved
    if captioner is None:
        raise RuntimeError(f'Missing captions at {path}; load the selected caption model.')
    record = {'sample_id': pair['sample_id'], 'index': pair['index'],
              'original_path': pair['source_path'], 'target_path': pair['target_path'],
              'image_sha256': expected_hashes,
              'captions': {'source': captioner.caption(images[0]),
                           'target': captioner.caption(images[1])},
              'caption_config': expected_config, 'caption_model': captioner.metadata,
              'created_at_utc': utc_stamp()}
    write_json(path, record)
    return record


def read_preprocessed_pair(pair, transform):
    with Image.open(pair['source_path']) as image:
        source = transform(image)
    with Image.open(pair['target_path']) as image:
        target = transform(image)
    return source, target


def export_caption_table(run_dir):
    folder = Path(run_dir) / 'captions'
    if folder.is_dir():
        rows = (json.loads(p.read_text(encoding='utf-8')) for p in sorted(folder.glob('*.json')))
        write_jsonl_atomic(Path(run_dir) / 'captions.jsonl', rows)

remember_code('generated_captions')


In [ ]:
BACKBONE_MAP = {'B16': ClipB16FeatureExtractor, 'B32': ClipB32FeatureExtractor, 'Laion': ClipLaionFeatureExtractor}

def set_foa_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class QuietEnsembleFeatureExtractor(EnsembleFeatureExtractor_ot):
    """Keep author's forward/K-means defaults; disable logging inside clustering itself."""
    def get_cluster_center(self, embedding_img, device):
        # Upstream source K-means has no iteration limit; reject NaNs before its loop.
        if embedding_img.ndim != 2 or not torch.isfinite(embedding_img).all():
            raise FloatingPointError('Non-finite/invalid source embeddings before K-means.')
        _, centers = kmeans(X=embedding_img, num_clusters=self.cluster_number,
                            distance='euclidean', device=device, tqdm_flag=False)
        if not torch.isfinite(centers).all():
            raise FloatingPointError('Non-finite source K-means centers.')
        return centers.to(device)

def build_foa_ensemble(device, backbones, cluster_number, prune_text):
    models = []
    for name in backbones:
        model = BACKBONE_MAP[name]()
        # global_local_features only accesses CLIP vision_model, never these text modules.
        # Remove before .to(device) to avoid allocating unused text weights on T4.
        if prune_text:
            for attr in ('text_model', 'text_projection'):
                if hasattr(model.model, attr):
                    delattr(model.model, attr)
        model = model.eval().requires_grad_(False).to(device)
        original_features = model.global_local_features
        use_autocast = bool(FOA_AUTOCAST)
        def precision_features(x, _forward=original_features, _autocast=use_autocast):
            with torch.autocast(device_type=x.device.type, dtype=torch.float16,
                                enabled=_autocast and x.device.type == 'cuda'):
                global_feature, local_feature = _forward(x)
            # K-means and OT must never run under autocast or consume half features.
            return global_feature.float(), local_feature.float()
        model.global_local_features = precision_features
        models.append(model)
    return QuietEnsembleFeatureExtractor(models, cluster_number=cluster_number), models


FOA_EXTRACTOR = None
FOA_MODELS = None
FOA_MODEL_SIGNATURE = None
ATTENTION_BACKEND = None

def ensure_foa_models():
    global FOA_EXTRACTOR, FOA_MODELS, FOA_MODEL_SIGNATURE
    signature = (ATTACK_DEVICE, tuple(PAPER_BACKBONES), PAPER_CLUSTER_NUMBER, PRUNE_TEXT_BRANCH, FOA_AUTOCAST)
    if FOA_EXTRACTOR is not None and signature != FOA_MODEL_SIGNATURE:
        raise RuntimeError('FOA model/device settings changed; restart session before loading another ensemble.')
    if FOA_EXTRACTOR is None:
        with preserve_rng():
            FOA_EXTRACTOR, FOA_MODELS = build_foa_ensemble(
                torch.device(ATTACK_DEVICE), PAPER_BACKBONES, PAPER_CLUSTER_NUMBER, PRUNE_TEXT_BRANCH)
            FOA_MODEL_SIGNATURE = signature
    return FOA_EXTRACTOR, FOA_MODELS

remember_code('foa_models')


In [ ]:
# Cell 6 — FOA loss. This is the paper loss copied into an editable cell.
class FOAOptimalTransportLoss(nn.Module):
    """Original FOA global/local OT loss with its dynamic ensemble weighting (K=10 by default)."""
    def __init__(self, extractors: List[nn.Module], cluster_number: int = PAPER_CLUSTER_NUMBER):
        """Initialize FOA state for the frozen surrogate extractors and local cluster count."""
        super().__init__(); self.extractors = nn.ModuleList(extractors); self.cluster_number = cluster_number
        self.ground_truth, self.ground_truth_local, self.previous_loss_list = [], [], []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor) -> None:
        """Extract target global features and K-means local centers as FOA ground truth."""
        self.ground_truth.clear(); self.ground_truth_local.clear()
        for model in self.extractors:
            global_feature, local_feature = model.global_local_features(x.to(x.device))
            centers = self.cluster_centers(local_feature.squeeze(0), x.device).unsqueeze(0)
            self.ground_truth.append(global_feature); self.ground_truth_local.append(centers)

    def cluster_centers(self, embeddings: Tensor, device: torch.device) -> Tensor:
        """Compute original Euclidean K-means local centers (100 iteration limit)."""
        if embeddings.ndim != 2 or not torch.isfinite(embeddings).all():
            raise FloatingPointError('Non-finite/invalid target embeddings before K-means.')
        _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device, iter_limit=100, tqdm_flag=False)
        if not torch.isfinite(centers).all(): raise FloatingPointError('Non-finite target K-means centers.')
        return centers.to(device)

    def optimal_transport(self, source: Tensor, target: Tensor) -> Tensor:
        """Return FOA cosine-similarity optimal-transport score using original Sinkhorn settings."""
        similarity = torch.einsum('md,nd->mn', F.normalize(source, dim=1), F.normalize(target, dim=1)).contiguous()
        u = torch.full((source.shape[0],), 1 / source.shape[0], dtype=similarity.dtype, device=similarity.device)
        v = torch.full((target.shape[0],), 1 / target.shape[0], dtype=similarity.dtype, device=similarity.device)
        with torch.no_grad():
            kernel = torch.exp(-(1 - similarity) / 0.1)
            transport = self.sinkhorn(kernel, u, v)
        if not torch.isfinite(transport).all(): raise FloatingPointError('Non-finite FOA transport.')
        return torch.sum(transport * similarity)

    def sinkhorn(self, kernel: Tensor, u: Tensor, v: Tensor) -> Tensor:
        """Run the original 100-step, 1e-2-threshold Sinkhorn normalization."""
        r, c = torch.ones_like(u), torch.ones_like(v)
        for _ in range(100):
            old_r = r; r = u / (kernel @ c.unsqueeze(-1)).squeeze(-1); c = v / (kernel.T @ r.unsqueeze(-1)).squeeze(-1)
            if (r - old_r).abs().mean().item() < 1e-2: break
        return torch.outer(r, c) * kernel

    def forward(self, global_features: Dict[int, Tensor], local_features: Dict[int, Tensor]) -> Tensor:
        """Compute original FOA global OT + 0.2 local OT and dynamic model weights."""
        losses = [self.optimal_transport(self.ground_truth[i], global_features[i].unsqueeze(0)) + 0.2 * self.optimal_transport(self.ground_truth_local[i].squeeze(0), local_features[i].squeeze(0)) for i in range(len(self.extractors))]
        if not self.previous_loss_list: self.previous_loss_list = [loss.detach() for loss in losses]
        weights = np.array([loss.item() / (prev.item() + 1e-8) for loss, prev in zip(losses, self.previous_loss_list)])
        with np.errstate(over='ignore', invalid='ignore'):
            weights = np.exp(weights) / np.exp(weights).sum() * len(weights)
        if not np.isfinite(weights).all(): raise FloatingPointError('FOA dynamic weights overflowed.')
        self.previous_loss_list = [loss.detach() for loss in losses]
        return sum(float(weights[i]) * loss for i, loss in enumerate(losses))

remember_code('foa_loss')


In [ ]:
"""Fixed-area hotspot ranking with the original IoU fallback."""



def compute_box_iou(box1, box2):
    # box: (x_min, x_max, y_min, y_max) inclusive
    x1_min, x1_max, y1_min, y1_max = box1
    x2_min, x2_max, y2_min, y2_max = box2

    inter_x_min = max(x1_min, x2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_min = max(y1_min, y2_min)
    inter_y_max = min(y1_max, y2_max)

    inter_w = max(0, inter_x_max - inter_x_min + 1)
    inter_h = max(0, inter_y_max - inter_y_min + 1)
    inter_area = inter_w * inter_h

    area1 = (x1_max - x1_min + 1) * (y1_max - y1_min + 1)
    area2 = (x2_max - x2_min + 1) * (y2_max - y2_min + 1)

    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def get_top_k_boxes_fixed_area(attention_map, target_ratio, k=1, iou_threshold=0.3, area_tolerance=0.05):
    """
    Find top-k boxes with maximum attention score that have area approximately equal to target_ratio * total_area.
    """
    H, W = attention_map.shape
    total_cells = H * W
    target_cells = int(target_ratio * total_cells)
    if target_cells < 1:
        target_cells = 1

    # When target ratio is effectively the full map, force search to stay near full area
    near_full = target_ratio >= 0.999
    area_lower = total_cells if near_full else max(1, int(target_cells * (1 - area_tolerance)))
    area_upper = total_cells if near_full else min(total_cells, int(target_cells * (1 + area_tolerance)))

    integral = np.cumsum(np.cumsum(attention_map, axis=0), axis=1)

    def get_sum(r1, c1, r2, c2):
        res = integral[r2, c2]
        if r1 > 0:
            res -= integral[r1 - 1, c2]
        if c1 > 0:
            res -= integral[r2, c1 - 1]
        if r1 > 0 and c1 > 0:
            res += integral[r1 - 1, c1 - 1]
        return res

    candidates = []

    for h in range(1, H + 1):
        w = int(round(target_cells / h))
        if w < 1:
            w = 1
        if w > W:
            w = W

        # Aspect ratio constraint to prevent elongated boxes
        if h / w > 3.0 or w / h > 3.0:
            continue

        area = h * w
        # Enforce area closeness to target to avoid undersized boxes dominating
        if area < area_lower or area > area_upper:
            continue

        for r in range(H - h + 1):
            for c in range(W - w + 1):
                current_sum = get_sum(r, c, r + h - 1, c + w - 1)
                # current_mean = current_sum / area
                candidates.append((current_sum, (c, c + w - 1, r, r + h - 1)))

    # Sort by score descending
    candidates.sort(key=lambda x: x[0], reverse=True)

    # Select top-k with IoU filtering
    selected_boxes = []
    for _, box in candidates:
        if len(selected_boxes) >= k:
            break

        # Check IoU with already selected boxes
        is_valid = True
        for selected_box in selected_boxes:
            if compute_box_iou(box, selected_box) > iou_threshold:
                is_valid = False
                break

        if is_valid:
            selected_boxes.append(box)

    # Fallback: if we could not gather k boxes under the IoU cap,
    # fill the rest with the next best candidates even if they overlap.
    if len(selected_boxes) < k and candidates:
        for _, box in candidates:
            if len(selected_boxes) >= k:
                break
            # Avoid exact duplicates
            if any(box == chosen for chosen in selected_boxes):
                continue
            selected_boxes.append(box)

    return selected_boxes


def build_schedule(attention_map, num_phases=10, top_k=3, iou_threshold=0.3):
    attention_map = np.asarray(attention_map)
    if attention_map.ndim != 2 or not attention_map.size or not np.isfinite(attention_map).all():
        raise ValueError("Attention must be a nonempty finite 2D map")
    if num_phases < 1 or top_k < 1 or not 0 <= iou_threshold <= 1:
        raise ValueError("Invalid hotspot schedule parameters")
    grid_h, grid_w = attention_map.shape
    schedule = []
    for i in range(num_phases):
        ratio = (i + 1) / num_phases
        boxes = get_top_k_boxes_fixed_area(attention_map, ratio, k=top_k, iou_threshold=iou_threshold)
        for j, (xmin, xmax, ymin, ymax) in enumerate(boxes):
            top = np.clip(ymin / grid_h, 0.0, 1.0)
            left = np.clip(xmin / grid_w, 0.0, 1.0)
            height = np.clip((ymax - ymin + 1) / grid_h, 0.0, 1.0 - top)
            width = np.clip((xmax - xmin + 1) / grid_w, 0.0, 1.0 - left)
            schedule.append(
                dict(
                    threshold=ratio,
                    box=tuple(map(float, (top, left, height, width))),
                    phase_group=i + 1,
                    sub_phase=j + 1,
                )
            )
    if not schedule:
        raise ValueError("No hotspot candidates found")
    return schedule

class RegionBiasedRandomResizedCrop(nn.Module):
    """
    RandomResizedCrop with biased sampling of hotspot region.
    """

    def __init__(
        self,
        size,
        scale=(0.8, 1.0),
        hotspot_rect: tuple[int, int, int, int] = None,
        hotspot_prob: float = 0.8,
        antialias=True,
    ):
        super().__init__()
        # Use standard RandomResizedCrop internally
        self.standard_cropper = transforms.RandomResizedCrop(size=size, scale=scale, antialias=antialias)

        if hotspot_rect is None:
            # If hotspot is not provided, always use standard crop
            self.hotspot_prob = 0.0
        else:
            self.hotspot_prob = hotspot_prob
            self.top, self.left, self.height, self.width = hotspot_rect

    def forward(self, img: torch.Tensor) -> torch.Tensor:
        # With p_hot probability, crop from the hotspot region
        if random.random() < self.hotspot_prob:
            # 1. Crop the hotspot region from the original image
            img_hotspot = TF.crop(img, self.top, self.left, self.height, self.width)

            # 2. Apply standard RandomResizedCrop to the cropped hotspot region
            #    (If the hotspot is too small to crop, an error may occur, so exception handling)
            try:
                return self.standard_cropper(img_hotspot), "hotspot"
            except ValueError:
                # If the hotspot is too small to crop, just resize
                return TF.resize(img_hotspot, self.standard_cropper.size, antialias=True), "hotspot"
        else:
            # With the remaining probability, use standard RandomResizedCrop on the entire image
            return self.standard_cropper(img), "global"


def box_to_pixels(box, height, width):
    top, left, h, w = map(float, box)
    if not all(math.isfinite(x) for x in (top, left, h, w)):
        raise ValueError('Non-finite hotspot box.')
    t, l = max(0, min(height - 1, int(top * height))), max(0, min(width - 1, int(left * width)))
    ph, pw = max(1, int(h * height)), max(1, int(w * width))
    return t, l, min(ph, height - t), min(pw, width - l)

def schedule_entry(schedule, step, steps):
    if not schedule or steps < len(schedule) or not 0 <= step < steps:
        raise ValueError('Need enough steps for every actual schedule entry.')
    block = steps // len(schedule)
    index = min(step // block, len(schedule) - 1)
    return index, schedule[index]

class TracedSourceCrop(nn.Module):
    """SAGA hotspot crop or FOA full-image crop, with exact random crop rectangle recorded."""
    def __init__(self, size, scale, hotspot_rect=None):
        super().__init__()
        self.hotspot_rect = hotspot_rect
        self.saga_cropper = RegionBiasedRandomResizedCrop(size=size, scale=scale,
                                                        hotspot_rect=hotspot_rect, hotspot_prob=1.0)
        self.last_rectangle = None

    def forward(self, image):
        # Preserve SAGA's Python-RNG draw for its hotspot branch; baseline does not draw it.
        if self.hotspot_rect is None:
            top, left, height, width = 0, 0, image.shape[-2], image.shape[-1]
        else:
            random.random()
            top, left, height, width = self.hotspot_rect
        region = TF.crop(image, top, left, height, width)
        standard = self.saga_cropper.standard_cropper
        t, l, h, w = transforms.RandomResizedCrop.get_params(region, standard.scale, standard.ratio)
        self.last_rectangle = (top + t, left + l, h, w)
        return TF.resized_crop(region, t, l, h, w, standard.size,
                               interpolation=standard.interpolation, antialias=standard.antialias)

def checkpoint_steps(steps):
    interval = max(1, steps // 10)
    return sorted({0, steps, *range(interval, steps + 1, interval)})

remember_code('saga_regions')


In [ ]:
DEBUG = os.environ.get("DEBUG", "0") == "1"

STOP_WORDS = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
}


def clean_token(token: str) -> str:
    """Lowercase, strip, and drop punctuation to normalize tokens."""
    return token.strip().lower().translate(str.maketrans("", "", string.punctuation))


def is_content_word(token: str) -> bool:
    """Return True if token is non-empty after cleaning and not a stop word."""
    cleaned = clean_token(token)
    return bool(cleaned) and cleaned not in STOP_WORDS


def filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:
    """
    Return indices of tokens to use. If use_content_words is True, keep only content
    words; otherwise return all indices. Falls back to all tokens if filtering empties.
    """
    if not use_content_words:
        return list(range(len(token_ids)))

    logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")
    valid_indices = []
    for idx, token_id in enumerate(token_ids):
        token_str = tokenizer.decode([token_id], skip_special_tokens=False)
        if is_content_word(token_str):
            valid_indices.append(idx)
        elif DEBUG:
            logger.debug(f"Filtering out token '{token_str}' at index {idx}")

    if not valid_indices:
        return list(range(len(token_ids)))

    return valid_indices


def attention_layout(input_ids, concept_start, concept_length, image_token_id, vision_start_id, vision_end_id, grid_shape):
    ids = input_ids.reshape(-1)
    image_positions = torch.nonzero(ids == image_token_id, as_tuple=False).flatten()
    starts = torch.nonzero(ids == vision_start_id, as_tuple=False).flatten()
    ends = torch.nonzero(ids == vision_end_id, as_tuple=False).flatten()
    if len(starts) != 1 or len(ends) != 1 or len(image_positions) != grid_shape[0] * grid_shape[1]:
        raise ValueError('Expected one image and matching image-token grid.')
    expected = torch.arange(int(starts[0]) + 1, int(ends[0]), device=ids.device)
    if not torch.equal(image_positions, expected):
        raise ValueError('Vision tokens must be contiguous and exclude marker tokens.')
    if concept_length < 1 or concept_start <= int(ends[0]) or concept_start + concept_length != len(ids):
        raise ValueError('Concept token interval must be the appended assistant text.')
    return image_positions

def normalize_concept_attention(token_to_vision):
    values = token_to_vision.detach().float()
    if values.ndim != 2 or not values.numel() or not torch.isfinite(values).all() or (values < 0).any():
        raise ValueError('Invalid concept-to-vision attention weights.')
    masses = values.sum(-1)
    if (masses <= 1e-12).any():
        raise ValueError('Concept has zero image-attention mass; not substituting a map.')
    probability = (values / masses[:, None]).mean(0)
    probability = probability / probability.sum()
    minimum, maximum = probability.min(), probability.max()
    minmax = (probability - minimum) / (maximum - minimum + 1e-8)
    return probability, minmax, masses


from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb, repeat_kv

class _AttentionReady(Exception):
    """Private control flow: no later layers are needed for a salient-region map."""

def selected_causal_vision_rows(module, hidden_states, position_embeddings, query_rows, vision_positions):
    """Exact eager equations for selected rows, including all-key head softmax.

    This function receives a single unpadded, uncached image/text sequence.
    Do not softmax only image keys: that would change relative head weighting.
    """
    if hidden_states.shape[0] != 1:
        raise ValueError('Attention capture requires one unpadded sequence.')
    shape = (*hidden_states.shape[:-1], -1, module.head_dim)
    q = module.q_norm(module.q_proj(hidden_states).view(shape)).transpose(1, 2)
    k = module.k_norm(module.k_proj(hidden_states).view(shape)).transpose(1, 2)
    q, k = apply_rotary_pos_emb(q, k, *position_embeddings)
    q = q.index_select(2, query_rows)
    k = repeat_kv(k, module.num_key_value_groups)
    scores = torch.matmul(q, k.transpose(2, 3)) * module.scaling
    future = torch.arange(k.shape[2], device=k.device)[None, :] > query_rows[:, None]
    scores = scores.masked_fill(future[None, None], torch.finfo(scores.dtype).min)
    weights = F.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)
    return weights[0].index_select(2, vision_positions).float().mean(0)

class ConceptAttentionExtractor:
    """SDPA prefix, selected Q/K rows at layer 29, early exit, per-image vision cache."""
    def __init__(self, resolved_revision=None):
        self.device = torch.device(ATTENTION_DEVICE)
        self.signature = self.current_signature()
        requested_revision = resolved_revision or ATTENTION_REVISION
        resolved_config = AutoConfig.from_pretrained(ATTENTION_MODEL_ID, revision=requested_revision)
        load_revision = getattr(resolved_config, '_commit_hash', None) or requested_revision
        self.processor = AutoProcessor.from_pretrained(ATTENTION_MODEL_ID, revision=load_revision)
        quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=['visual'])
        self.model = AutoModelForImageTextToText.from_pretrained(
            ATTENTION_MODEL_ID, revision=load_revision, torch_dtype=torch.float16,
            quantization_config=quantization, device_map={'': self.device.index},
            attn_implementation='sdpa', low_cpu_mem_usage=True, trust_remote_code=False).eval()
        self.model.requires_grad_(False)
        layers = self.model.model.language_model.layers
        if not 0 <= ATTENTION_LAYER < len(layers):
            raise ValueError('Attention layer index is unavailable.')
        self.attention_module = layers[ATTENTION_LAYER].self_attn
        self._lock = threading.RLock()
        self._prepared = None
        self.metadata = {'model_id': ATTENTION_MODEL_ID,
            'resolved_revision': getattr(self.model.config, '_commit_hash', None),
            'layer': ATTENTION_LAYER, 'dtype': 'float16', 'quantization': 'nf4_double',
            'device': str(self.device), 'input_res': ATTENTION_INPUT_RES,
            'mode': 'teacher_forced_concept_input_rows', 'attn_implementation': 'sdpa_prefix_selected_eager_rows',
            'early_exit': 'before selected self-attention value/output projection and MLP',
            'vision_cache_scope': 'same checkpoint image only', 'use_cache': False}

    @staticmethod
    def current_signature():
        return (ATTENTION_MODEL_ID, ATTENTION_REVISION, ATTENTION_DEVICE, ATTENTION_LAYER,
                ATTENTION_INPUT_RES, ATTENTION_PROMPT, MAX_CONCEPT_TOKENS, CONTENT_WORDS_ONLY)

    @contextlib.contextmanager
    def image_context(self, image):
        # One thread owns model hooks, the processor and its mutable RoPE state.
        device_context = torch.cuda.device(self.device) if self.device.type == 'cuda' else contextlib.nullcontext()
        with self._lock, torch.inference_mode(), device_context:
            if self.current_signature() != self.signature:
                raise RuntimeError('Attention settings changed during a run; concept protocol must stay fixed.')
            if self.model.training:
                raise RuntimeError('Attention must run in eval mode without dropout/RNG draws.')
            if self._prepared is not None:
                raise RuntimeError('Nested image contexts are not supported.')
            image = image.convert('RGB').resize((ATTENTION_INPUT_RES, ATTENTION_INPUT_RES), Image.Resampling.BICUBIC)
            messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': ATTENTION_PROMPT}]}]
            prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            base = self.processor(text=[prompt], images=[image], do_resize=False, return_tensors='pt')
            self._prepared = {key: value.to(self.device) for key, value in base.items()}
            visual = self.model.model.visual
            original_forward = visual.forward
            had_override = 'forward' in visual.__dict__
            cache = {}
            def cached_visual(*args, **kwargs):
                if 'features' not in cache:
                    cache['features'] = original_forward(*args, **kwargs)
                return cache['features']
            visual.forward = cached_visual
            try:
                yield self
            finally:
                if had_override:
                    visual.forward = original_forward
                else:
                    del visual.forward
                self._prepared = None
                cache.clear()
                # An early exit can leave RoPE deltas set; every concept computes fresh positions.
                self.model.model.rope_deltas = None

    @torch.inference_mode()
    def extract_prepared(self, concept):
        if self.current_signature() != self.signature:
            raise RuntimeError('Attention settings changed during a run; restart with a fixed configuration.')
        if self._prepared is None:
            raise RuntimeError('Use image_context before extract_prepared.')
        if not isinstance(concept, str) or not concept.strip():
            raise ValueError('Concept must be nonempty text.')
        inputs = dict(self._prepared)
        tokenizer = self.processor.tokenizer
        concept_ids = tokenizer.encode(concept, add_special_tokens=False)
        if not concept_ids or len(concept_ids) > MAX_CONCEPT_TOKENS:
            raise ValueError(f'Concept needs 1..{MAX_CONCEPT_TOKENS} tokens; no silent truncation.')
        if any(token in tokenizer.all_special_ids for token in concept_ids):
            raise ValueError('Concept must not contain special control tokens.')
        start = inputs['input_ids'].shape[1]
        suffix = torch.tensor([concept_ids], device=self.device, dtype=inputs['input_ids'].dtype)
        inputs['input_ids'] = torch.cat((inputs['input_ids'], suffix), dim=1)
        inputs['attention_mask'] = torch.ones_like(inputs['input_ids'])
        inputs.pop('position_ids', None); inputs.pop('token_type_ids', None)
        grid = inputs['image_grid_thw'][0].cpu()
        merge = self.processor.image_processor.merge_size
        if int(grid[0]) != 1 or int(grid[1]) % merge or int(grid[2]) % merge:
            raise ValueError('Expected one still-image divisible merged grid.')
        shape = (int(grid[1]) // merge, int(grid[2]) // merge)
        config = self.model.config
        vision = attention_layout(inputs['input_ids'][0], start, len(concept_ids),
            config.image_token_id, config.vision_start_token_id, config.vision_end_token_id, shape)
        offsets = filter_generated_tokens(tokenizer, concept_ids, CONTENT_WORDS_ONLY)
        queries = torch.tensor([start + offset for offset in offsets], device=self.device)
        captured = {}
        def capture(module, args, kwargs):
            if kwargs.get('past_key_values') is not None:
                raise RuntimeError('KV cache is forbidden during attention capture.')
            hidden = kwargs.get('hidden_states', args[0] if args else None)
            position_embeddings = kwargs.get('position_embeddings')
            if hidden is None or position_embeddings is None:
                raise RuntimeError('Unexpected Qwen self-attention signature; install pinned Transformers.')
            captured['weights'] = selected_causal_vision_rows(module, hidden, position_embeddings, queries, vision).cpu()
            raise _AttentionReady()
        handle = self.attention_module.register_forward_pre_hook(capture, with_kwargs=True)
        self.model.model.rope_deltas = None
        try:
            try:
                self.model.model(**inputs, use_cache=False, output_attentions=False,
                                 output_hidden_states=False, return_dict=True)
            except _AttentionReady:
                pass
        finally:
            handle.remove()
        if 'weights' not in captured:
            raise RuntimeError('Selected attention layer was not reached.')
        probability, minmax, masses = normalize_concept_attention(captured['weights'])
        return {'probability': probability.reshape(shape).numpy(), 'minmax': minmax.reshape(shape).numpy(),
            'metadata': {'concept': concept, 'token_ids': concept_ids, 'selected_offsets': offsets,
                'selected_token_texts': [tokenizer.decode([concept_ids[i]]) for i in offsets],
                'concept_query_rows': queries.cpu().tolist(), 'vision_positions': vision.cpu().tolist(),
                'grid_shape': list(shape), 'mean_raw_vision_mass': float(masses.mean()),
                'method': 'teacher_forced_concept_input_rows', 'layer': ATTENTION_LAYER,
                'backend': 'sdpa_prefix_selected_eager_rows', 'early_exit': True}}

    def extract(self, image, concept):
        with self.image_context(image):
            return self.extract_prepared(concept)

def ensure_attention_backend(resolved_revision=None):
    global ATTENTION_BACKEND
    if ATTENTION_BACKEND is not None and ConceptAttentionExtractor.current_signature() != ATTENTION_BACKEND.signature:
        raise RuntimeError('Qwen configuration changed; restart session before reloading.')
    if (ATTENTION_BACKEND is not None and resolved_revision is not None
            and ATTENTION_BACKEND.metadata.get('resolved_revision') != resolved_revision):
        raise RuntimeError('Cached Qwen revision differs from the resumed run; restart the session.')
    if ATTENTION_BACKEND is None:
        with preserve_rng():
            ATTENTION_BACKEND = ConceptAttentionExtractor(resolved_revision=resolved_revision)
    return ATTENTION_BACKEND

remember_code('concept_attention')


In [ ]:
def probability_overlay(image, probability, axis, title, vmax, rectangle=None):
    axis.imshow(image)
    axis.imshow(probability, extent=(0, image.width, image.height, 0), interpolation='bilinear',
                cmap='inferno', vmin=0, vmax=vmax, alpha=0.55)
    if rectangle is not None:
        top, left, height, width = rectangle
        axis.add_patch(Rectangle((left, top), width, height, fill=False, edgecolor='cyan', linewidth=1.4))
    axis.set_title('\n'.join(textwrap.fill(line, width=42) for line in title.splitlines()), fontsize=9)
    axis.axis('off')

def preview_checkpoint(image_path, results, step, hotspot=None, crop=None, handle=None, scales=None):
    image = Image.open(image_path).convert('RGB')
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image); axes[0].set_title(f'Step {step}: adversarial'); axes[0].axis('off')
    for rect, color in ((hotspot, 'cyan'), (crop, 'lime')):
        if rect is not None:
            t, l, h, w = rect
            axes[0].add_patch(Rectangle((l, t), w, h, fill=False, edgecolor=color, linewidth=1.3))
    for axis, label in zip(axes[1:], ('source', 'target')):
        result = results[label]
        # Initial-image maximum is frozen for live playback; final timeline uses all-step maximum.
        vmax = scales[label] if scales is not None else 1.0
        probability_overlay(image, result['probability'], axis,
                            f'{label}: {result["metadata"]["concept"]}\nFixed scale 0..{vmax:.4g}', vmax, hotspot)
    fig.tight_layout()
    if handle is None:
        handle = display(fig, display_id=True)
    else:
        handle.update(fig)
    plt.close(fig)
    return handle

def plot_attention_timeline(sample_dir, attention_root=None, figure_dir=None, show=True):
    sample_dir = Path(sample_dir)
    attention_root = sample_dir / 'attention' if attention_root is None else Path(attention_root)
    figure_dir = sample_dir / 'figures' if figure_dir is None else Path(figure_dir)
    figure_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = attention_root / 'concepts.json'
    if not metadata_path.is_file():
        return []
    concepts = json.loads(metadata_path.read_text(encoding='utf-8'))
    geometry_path = sample_dir / 'attention' / 'checkpoint_metrics.jsonl'
    geometry = {}
    if geometry_path.is_file():
        geometry = {row['step']: row for row in
                    (json.loads(line) for line in geometry_path.read_text(encoding='utf-8').splitlines() if line.strip())}
    outputs = []
    for label, concept in concepts.items():
        files = sorted(attention_root.glob(f'step_*/{label}.npz'))
        if not files:
            continue
        steps = [int(path.parent.name.split('_')[-1]) for path in files]
        maps = [np.load(path)['probability'] for path in files]
        vmax = max(float(values.max()) for values in maps)
        limit = max(float(np.abs(values - maps[0]).max()) for values in maps) or 1e-12
        fig, axes = plt.subplots(2, len(files), figsize=(max(6, 2.3 * len(files)), 5), squeeze=False)
        for column, (step, probability) in enumerate(zip(steps, maps)):
            with Image.open(sample_dir / 'checkpoints' / f'step_{step:05d}.png') as img:
                image = img.convert('RGB')
            region = geometry.get(step, {})
            hotspot = region.get('hotspot')
            if step == 0 and region.get('initial_hotspot_box') is not None:
                hotspot = box_to_pixels(region['initial_hotspot_box'], image.height, image.width)
            probability_overlay(image, probability, axes[0, column], f'Step {step}', vmax, hotspot)
            if region.get('last_crop') is not None:
                t, l, h, w = region['last_crop']
                axes[0, column].add_patch(Rectangle((l, t), w, h, fill=False, edgecolor='lime', linewidth=.9))
            difference = axes[1, column].imshow(probability - maps[0], cmap='coolwarm', vmin=-limit, vmax=limit)
            axes[1, column].set_title('Δ probability vs step 0', fontsize=8); axes[1, column].axis('off')
        fig.suptitle(f'{label}: {concept}\nShared probability scale: 0..{vmax:.5g}')
        fig.colorbar(axes[0, -1].images[1], ax=list(axes[0]), shrink=0.65, label='Attention probability')
        fig.colorbar(difference, ax=list(axes[1]), shrink=0.65, label='Δ probability')
        path = figure_dir / f'timeline_{label}.png'
        fig.savefig(path, dpi=140, bbox_inches='tight'); outputs.append(str(path))
        if show:
            display(fig)
        plt.close(fig)
    trace_path = sample_dir / 'trace.csv'
    checkpoints_path = attention_root / 'checkpoint_metrics.jsonl'
    if trace_path.is_file() and checkpoints_path.is_file():
        with trace_path.open(encoding='utf-8', newline='') as stream:
            trace = list(csv.DictReader(stream))
        records = [json.loads(line) for line in checkpoints_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
        if trace:
            axes[0].plot([int(r['step']) for r in trace], [float(r['objective']) for r in trace])
            axes[2].plot([int(r['step']) for r in trace], [float(r['linf_internal_255']) for r in trace], label='Internal float')
        for label in concepts:
            values = [r for r in records if label in r['concept_metrics']]
            if values:
                axes[1].plot([r['step'] for r in values],
                             [r['concept_metrics'][label]['initial_hotspot_mass'] for r in values], label=label)
        axes[2].plot([r['step'] for r in records], [r['linf_saved_255'] for r in records], 'o', label='PNG')
        for axis, title in zip(axes, ('FOA objective (maximize)', 'Mass in fixed initial guide hotspot', 'L∞ perturbation (0–255)')):
            axis.set_title(title); axis.set_xlabel('Step'); axis.grid(alpha=.25)
        axes[1].legend(); axes[2].legend(); fig.tight_layout()
        path = figure_dir / 'metrics.png'; fig.savefig(path, dpi=140); outputs.append(str(path))
        if show:
            display(fig)
        plt.close(fig)
    return outputs

remember_code('visualization')


In [ ]:
def hotspot_grid_mass(probability, box):
    height, width = probability.shape
    top, left, h, w = box_to_pixels(box, height, width)
    return float(probability[top:top + h, left:left + w].sum())

class CheckpointObserver:
    """Fresh guide barrier + bounded background observation on GPU 1.

    No preserve_rng in a background thread: restoring global RNG there would race
    with attack sampling. The inference-only worker performs no random operations.
    """
    def __init__(self, sample_dir, concepts, backend, *, live=False, initial_box=None, total_steps=STEPS, epsilon=PAPER_EPSILON):
        self.sample_dir = Path(sample_dir)
        self._fixed_concepts = dict(concepts)
        if 'guide' not in self._fixed_concepts:
            raise ValueError('A fixed guide concept is required.')
        self.concept_hashes = {label: concept_sha256(text) for label, text in self._fixed_concepts.items()}
        self.concepts = MappingProxyType(self._fixed_concepts.copy())
        self.token_signatures = {}
        self.observe_every = OBSERVE_ALL_CONCEPTS_EVERY
        self.epsilon = epsilon
        self.backend, self.live, self.initial_box = backend, live, initial_box
        self.total_steps = total_steps
        self.snapshot_steps = checkpoint_steps(total_steps)
        self.saved_steps, self.finished_steps = set(), set()
        self.initial_probabilities, self.live_scales = {}, {}
        self.preview_handle = None
        self.attention_seconds = self.guide_wait_seconds = 0.0
        self.pending = None
        self.closed = False
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix='qwen_attention') if ASYNC_OBSERVATION else None
        (self.sample_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
        (self.sample_dir / 'attention').mkdir(parents=True, exist_ok=True)
        write_json(self.sample_dir / 'attention' / 'concepts.json', dict(self.concepts))
        write_json(self.sample_dir / 'attention' / 'concept_fingerprints.json', self.concept_hashes)

    def validate_result(self, label, result):
        text = self._fixed_concepts[label]
        metadata = dict(result['metadata'])
        if metadata.get('concept') != text:
            raise ValueError(f'Backend used a different concept for {label}.')
        probability, minmax = np.asarray(result['probability']), np.asarray(result['minmax'])
        if (probability.ndim != 2 or not probability.size or probability.shape != minmax.shape
                or not np.isfinite(probability).all() or probability.min() < 0
                or not np.isclose(probability.sum(), 1, atol=1e-5)
                or not np.isfinite(minmax).all() or minmax.min() < 0 or minmax.max() > 1 + 1e-5):
            raise ValueError('Invalid attention maps; refusing to guide an attack with them.')
        raw_mass = metadata.get('mean_raw_vision_mass', 0)
        if not math.isfinite(raw_mass) or raw_mass <= 0:
            raise ValueError('Invalid raw image-attention mass.')
        required = ('token_ids', 'selected_offsets', 'concept_query_rows', 'vision_positions', 'grid_shape', 'method', 'layer')
        if any(key not in metadata for key in required):
            raise ValueError('Attention metadata must prove concept-token/image-token alignment.')
        ids, offsets = metadata['token_ids'], metadata['selected_offsets']
        rows, vision = metadata['concept_query_rows'], metadata['vision_positions']
        if (not ids or not offsets or len(rows) != len(offsets) or sorted(set(offsets)) != offsets
                or any(type(i) is not int or not 0 <= i < len(ids) for i in offsets)
                or len(vision) != probability.size or tuple(metadata['grid_shape']) != probability.shape
                or not vision or min(rows) <= max(vision)
                or len(set(row - offset for row, offset in zip(rows, offsets))) != 1):
            raise ValueError('Invalid concept-query or image-key token alignment.')
        signature = attention_token_fingerprint(metadata)
        if label in self.token_signatures and self.token_signatures[label] != signature:
            raise ValueError(f'Concept tokens/grid changed between checkpoints: {label}.')
        self.token_signatures[label] = signature
        metadata.update(concept_sha256=self.concept_hashes[label], token_signature_sha256=signature)
        return {**result, 'metadata': metadata}

    def labels_at(self, step):
        ordinal = self.snapshot_steps.index(step)
        all_labels = step in (0, self.total_steps) or (
            self.observe_every > 0 and ordinal % self.observe_every == 0)
        return ['guide'] + [label for label in self.concepts if label != 'guide' and all_labels]

    def _work(self, path, step, clean_array, hotspot, crop, guide_future):
        started = time.perf_counter()
        try:
            with Image.open(path) as image:
                image = image.convert('RGB')
            linf = int(np.abs(np.asarray(image).astype(np.int16) - clean_array).max())
            if linf > math.ceil(self.epsilon):
                raise AssertionError('Checkpoint PNG exceeds perturbation budget.')
            output_dir = self.sample_dir / 'attention' / f'step_{step:05d}'
            output_dir.mkdir(exist_ok=False)
            results, by_text = {}, {}
            manager = self.backend.image_context(image) if hasattr(self.backend, 'image_context') else contextlib.nullcontext()
            with manager:
                for label in self.labels_at(step):
                    text = self.concepts[label]
                    if text not in by_text:
                        by_text[text] = (self.backend.extract_prepared(text) if hasattr(self.backend, 'extract_prepared')
                                         else self.backend.extract(image, text))
                    result = self.validate_result(label, by_text[text])
                    probability = np.asarray(result['probability'])
                    if not np.isfinite(probability).all() or probability.min() < 0 or not np.isclose(probability.sum(), 1, atol=1e-5):
                        raise ValueError('Backend returned an invalid probability map.')
                    results[label] = result
                    np.savez_compressed(output_dir / f'{label}.npz', probability=probability, minmax=result['minmax'])
                    write_json(output_dir / f'{label}.json', {**result['metadata'], 'checkpoint_sha256': sha256_file(path),
                        'map_sha256': sha256_file(output_dir / f'{label}.npz')})
                    if label == 'guide':
                        if self.initial_box is None:
                            self.initial_box = build_schedule(result['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)[0]['box']
                        # A fresh saved-PNG map is ready. GPU 0 may start its next update now.
                        guide_future.set_result({'guide': result})
            metrics = {}
            for label, result in results.items():
                probability = result['probability']
                self.initial_probabilities.setdefault(label, probability.copy())
                self.live_scales.setdefault(label, float(probability.max()))
                initial = self.initial_probabilities[label]
                if initial.shape != probability.shape:
                    raise ValueError('Attention grid changed between checkpoints.')
                metrics[label] = {'initial_hotspot_mass': hotspot_grid_mass(probability, self.initial_box),
                    'l1_vs_initial': float(np.abs(probability - initial).sum()),
                    'mean_raw_vision_mass': result['metadata']['mean_raw_vision_mass']}
            record = {'step': step, 'linf_saved_255': linf, 'hotspot': hotspot, 'last_crop': crop,
                'initial_hotspot_box': self.initial_box, 'concept_metrics': metrics,
                'observed_labels': list(results), 'checkpoint_sha256': sha256_file(path),
                'guide_concept_sha256': self.concept_hashes['guide'],
                'guide_token_signature_sha256': self.token_signatures['guide']}
            with (self.sample_dir / 'attention' / 'checkpoint_metrics.jsonl').open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(record, allow_nan=False) + '\n')
            self.finished_steps.add(step)
            return results
        except BaseException as error:
            if not guide_future.done():
                guide_future.set_exception(error)
            raise
        finally:
            self.attention_seconds += time.perf_counter() - started

    def drain(self):
        if self.pending is not None:
            future, path, step, hotspot, crop = self.pending
            self.pending = None
            results = future.result()  # Propagate auxiliary failures; never silently discard.
            if self.live and 'source' in results and 'target' in results:
                self.preview_handle = preview_checkpoint(path, results, step, hotspot, crop,
                    self.preview_handle, self.live_scales)

    def capture(self, step, pixels, clean_pixels, *, hotspot=None, crop=None):
        if dict(self.concepts) != self._fixed_concepts:
            raise ValueError('Concept text changed inside a sample; start a new run instead.')
        if self.closed or step in self.saved_steps or step not in self.snapshot_steps:
            raise ValueError('Closed observer, duplicate or unexpected checkpoint.')
        self.drain()  # Bounded queue: at most one checkpoint owns Qwen memory.
        path = self.sample_dir / 'checkpoints' / f'step_{step:05d}.png'
        path.write_bytes(tensor_png_bytes(pixels))
        if not hasattr(self, 'clean_array'):
            self.clean_array = np.asarray(Image.open(io.BytesIO(tensor_png_bytes(clean_pixels)))).astype(np.int16)
        guide_future = Future()
        arguments = (path, step, self.clean_array, hotspot, crop, guide_future)
        started = time.perf_counter()
        if self.pool is None:
            future = Future()
            try:
                future.set_result(self._work(*arguments))
            except BaseException as error:
                future.set_exception(error)
        else:
            future = self.pool.submit(self._work, *arguments)
        self.pending = (future, path, step, hotspot, crop)
        results = guide_future.result()
        self.guide_wait_seconds += time.perf_counter() - started
        self.saved_steps.add(step)
        return results

    def close(self):
        if self.closed:
            return
        self.closed = True
        try:
            self.drain()
        finally:
            if self.pool is not None:
                self.pool.shutdown(wait=True, cancel_futures=False)

remember_code('observer')


In [ ]:
def foa_attention_attack(image_org, image_tgt, extractor, loss_fn, *, observer=None,
                         sample_dir=None, mode='dynamic', steps=STEPS,
                         epsilon=PAPER_EPSILON, alpha=PAPER_ALPHA, on_step=None,
                         source_crop=None, target_crop=None, on_phase=None):
    if mode not in {'baseline', 'fixed', 'dynamic'}:
        raise ValueError('Unknown attack mode.')
    if image_org.ndim != 4 or image_org.shape[0] != 1 or image_org.shape != image_tgt.shape:
        raise ValueError('FOA supports one equally sized source/target image pair.')
    if not torch.isfinite(image_org).all() or not torch.isfinite(image_tgt).all():
        raise ValueError('Non-finite input image.')
    if source_crop is None:
        source_crop = TracedSourceCrop(PAPER_INPUT_RES, PAPER_CROP_SCALE)
    if target_crop is None:
        target_crop = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
    if mode != 'baseline' and observer is None:
        raise ValueError('Attention-guided attack needs a checkpoint observer.')
    if observer is not None and (observer.total_steps != steps or observer.epsilon != epsilon):
        raise ValueError('Attack and observer must use the same steps/perturbation budget.')
    sample_dir = Path(sample_dir) if sample_dir is not None else None
    delta = torch.zeros_like(image_org, requires_grad=True)
    schedule, schedule_version = None, 0
    snapshots = set(checkpoint_steps(steps))
    trace_rows = []
    trace_handle = None
    if sample_dir is not None:
        sample_dir.mkdir(parents=True, exist_ok=True)
        trace_handle = (sample_dir / 'trace.csv').open('w', encoding='utf-8', newline='')
        trace_writer = csv.DictWriter(trace_handle, fieldnames=['step', 'objective', 'linf_internal_255',
            'schedule_version', 'schedule_index', 'phase_group', 'sub_phase', 'hotspot', 'crop', 'seconds'])
        trace_writer.writeheader(); trace_handle.flush()

    def update_schedule(results, completed_steps):
        nonlocal schedule, schedule_version
        schedule = build_schedule(results['guide']['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)
        if steps < len(schedule):
            raise ValueError(f'STEPS={steps} must visit {len(schedule)} schedule entries.')
        schedule_version = completed_steps
        if sample_dir is not None:
            write_json(sample_dir / 'schedules' / f'step_{completed_steps:05d}.json',
                       { 'computed_at_step': completed_steps, 'applies_from_update': completed_steps + 1, 'guide': observer.concepts['guide'],
                        'guide_concept_sha256': results['guide']['metadata']['concept_sha256'],
                        'guide_token_signature_sha256': results['guide']['metadata']['token_signature_sha256'],
                        'checkpoint_sha256': sha256_file(observer.sample_dir / 'checkpoints' / f'step_{completed_steps:05d}.png'),
                        'schedule': schedule, 'steps_per_item': steps // len(schedule),
                        'selection': 'absolute_step // (total_steps // actual_schedule_length); last receives remainder'})

    try:
        if observer is not None:
            if on_phase is not None: on_phase('attention', 0)
            results = observer.capture(0, image_org, image_org)
            if mode != 'baseline':
                update_schedule(results, 0)
            if on_phase is not None: on_phase('attack', 0)
        for step in range(steps):
            started = time.perf_counter()
            if mode == 'baseline':
                cropper, hotspot, schedule_index, entry = source_crop, None, -1, {}
            else:
                schedule_index, entry = schedule_entry(schedule, step, steps)
                hotspot = box_to_pixels(entry['box'], image_org.shape[-2], image_org.shape[-1])
                cropper = TracedSourceCrop(PAPER_INPUT_RES, PAPER_CROP_SCALE, hotspot)
            with torch.no_grad():
                loss_fn.set_ground_truth(target_crop(image_tgt))
            adv_image = image_org + delta
            # Full pass still runs first and advances FOA dynamic weights.
            # Grad graph is needed only when identity crop selects this full score.
            identity = isinstance(cropper, nn.Identity)
            with contextlib.nullcontext() if identity else torch.no_grad():
                global_features, local_features = extractor(adv_image)
                global_score = loss_fn(global_features, local_features)
            if identity:
                objective = global_score
            else:
                del global_features, local_features, global_score
                crop_global, crop_local = extractor(cropper(adv_image))
                objective = loss_fn(crop_global, crop_local)
            if not torch.isfinite(objective).all():
                raise FloatingPointError('Non-finite FOA objective.')
            gradient = torch.autograd.grad(objective * GRADIENT_SCALE, delta, create_graph=False)[0]
            if not torch.isfinite(gradient).all():
                raise FloatingPointError('Non-finite FOA gradient.')
            with torch.no_grad():
                delta.copy_((delta + alpha * gradient.sign()).clamp(-epsilon, epsilon))
            rectangle = getattr(cropper, 'last_rectangle', None)
            row = {'step': step + 1, 'objective': float(objective.detach()),
                   'linf_internal_255': float(delta.detach().abs().max()),
                   'schedule_version': schedule_version, 'schedule_index': schedule_index,
                   'phase_group': entry.get('phase_group', 0), 'sub_phase': entry.get('sub_phase', 0),
                   'hotspot': json.dumps(hotspot), 'crop': json.dumps(rectangle),
                   'seconds': time.perf_counter() - started}
            trace_rows.append(row)
            if trace_handle is not None:
                trace_writer.writerow(row)
            if on_step is not None and ((step + 1) % PROGRESS_INTERVAL == 0 or step + 1 == steps):
                on_step(step + 1)
            # Release full/crop forward graphs BEFORE Qwen checkpoint inference.
            del objective, gradient, adv_image
            if not identity:
                del crop_global, crop_local
            else:
                del global_features, local_features, global_score
            completed = step + 1
            if trace_handle is not None and completed in snapshots:
                trace_handle.flush()
            if observer is not None and completed in snapshots:
                if on_phase is not None: on_phase('attention', completed)
                results = observer.capture(completed, image_org + delta.detach(), image_org,
                                           hotspot=hotspot, crop=rectangle)
                if mode == 'dynamic' and completed < steps:
                    update_schedule(results, completed)
                if on_phase is not None: on_phase('attack' if completed < steps else 'finalizing_attention', completed)
        if float(delta.detach().abs().max()) > epsilon + 1e-5:
            raise AssertionError('Internal perturbation budget violated.')
        return ((image_org + delta.detach()) / 255.0).clamp(0, 1), trace_rows
    finally:
        try:
            if observer is not None:
                observer.close()
        finally:
            if trace_handle is not None:
                trace_handle.close()

remember_code('attack')


In [ ]:
def verify_baseline_parity():
    """CPU checks: same full/crop loss order and update as v1.2, RNG-neutral observers."""
    class MockExtractor(nn.Module):
        def forward(self, x):
            return {0: x.mean((2, 3)).squeeze(0)}, {0: x.mean((2, 3)).unsqueeze(1)}
    class MockLoss(nn.Module):
        def __init__(self):
            super().__init__(); self.calls = []
        def set_ground_truth(self, x):
            self.target = x.mean((2, 3)).detach(); self.calls.append('target')
        def forward(self, global_f, local_f):
            self.calls.append('loss')
            return -(global_f[0] - self.target.squeeze(0)).square().mean()
    source = (REPO_DIR / 'generate_adversarial_samples_foa_attack.py').read_text(encoding='utf-8')
    original_node = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == 'fgsm_attack')
    namespace = {'torch': torch, 'nn': nn, 'tqdm': lambda seq, **kw: seq,
                 'log_metrics': lambda *a, **kw: None, 'MainConfig': object, 'Optional': Optional, 'transforms': transforms}
    exec(compile(ast.Module(body=[original_node], type_ignores=[]), '<original-foa>', 'exec'), namespace)
    from types import SimpleNamespace
    x, y = torch.full((1, 3, 16, 16), 100.0), torch.full((1, 3, 16, 16), 130.0)
    for cropped in (False, True):
        def crop():
            return transforms.RandomResizedCrop(16, scale=PAPER_CROP_SCALE) if cropped else nn.Identity()
        config = SimpleNamespace(optim=SimpleNamespace(steps=3, epsilon=16, alpha=1),
                                 model=SimpleNamespace(use_source_crop=cropped))
        set_foa_seed(123)
        reference_loss = MockLoss()
        reference = namespace['fgsm_attack'](config, MockExtractor(), reference_loss, crop(), crop(), 0, x, y)
        set_foa_seed(123)
        ours_loss = MockLoss()
        ours, _ = foa_attention_attack(x, y, MockExtractor(), ours_loss, mode='baseline', steps=3,
                                      source_crop=crop(), target_crop=crop())
        assert torch.equal(reference, ours), 'Baseline changed FOA update.'
        assert reference_loss.calls == ours_loss.calls, 'Baseline changed FOA full/crop ordering.'
    set_foa_seed(42)
    with preserve_rng():
        random.random(); np.random.rand(); torch.rand(10)
    observed = (random.random(), np.random.rand(), torch.rand(3))
    set_foa_seed(42)
    expected = (random.random(), np.random.rand(), torch.rand(3))
    assert observed[:2] == expected[:2] and torch.equal(observed[2], expected[2])
    assert checkpoint_steps(300) == list(range(0, 301, 30))
    print('Baseline crop on/off, loss order, checkpoint schedule and RNG checks passed.')

if RUN_PARITY_CHECK:
    with preserve_rng():
        verify_baseline_parity()

remember_code('parity')


In [ ]:
def current_run_config():
    names = ['NUM_SAMPLES', 'STEPS', 'PAPER_EPSILON', 'PAPER_ALPHA', 'PAPER_INPUT_RES', 'PAPER_CROP_SCALE',
        'PAPER_BACKBONES', 'PAPER_CLUSTER_NUMBER', 'PAPER_SEED', 'ATTACK_DEVICE', 'ATTENTION_DEVICE',
        'ATTENTION_MODE', 'NUM_PHASES', 'HOTSPOT_TOP_K', 'HOTSPOT_IOU', 'GUIDE_CAPTION_ROLE',
        'PAIR_MANIFEST_PATH', 'TARGET_SUBDIR', 'PAIRING_STRICT', 'CAPTION_MODEL_ID',
        'CAPTION_MODEL_REVISION', 'CAPTION_DEVICE', 'CAPTION_PROMPT',
        'CAPTION_MAX_NEW_TOKENS', 'CAPTION_QUANTIZATION',
        'GUIDE_TEXT_OVERRIDE', 'GUIDE_OVERRIDES', 'EXTRA_CONCEPTS', 'EXTRA_CONCEPTS_BY_SAMPLE',
        'ATTENTION_MODEL_ID', 'ATTENTION_REVISION', 'ATTENTION_LAYER', 'ATTENTION_INPUT_RES',
        'MAX_CONCEPT_TOKENS', 'CONTENT_WORDS_ONLY', 'ATTENTION_PROMPT', 'PRUNE_TEXT_BRANCH',
        'FOA_AUTOCAST', 'GRADIENT_SCALE', 'OBSERVE_ALL_CONCEPTS_EVERY', 'ASYNC_OBSERVATION',
        'PLOT_FIRST_N_SAMPLES', 'SHOW_FINAL_PLOTS', 'LIVE_PREVIEW', 'CONTINUE_ON_ERROR', 'SAMPLE_START']
    return {name: str(globals()[name]) if isinstance(globals()[name], Path) else globals()[name]
            for name in names}

def pack_artifacts(directory):
    directory = Path(directory)
    archive = directory.with_suffix('.zip')
    temporary = Path(str(archive) + '.tmp')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL, allowZip64=True) as stream:
        for path in sorted(directory.rglob('*')):
            if path.is_file():
                stream.write(path, path.relative_to(directory).as_posix())
    temporary.replace(archive)
    return archive

def run_attention_experiment():
    global RUN_DIR
    validate_devices()
    # Default-device allocations in FOA stay on GPU 0; worker uses a thread-local GPU 1 context.
    torch.cuda.set_device(torch.device(ATTACK_DEVICE))
    pairs, caption_origin = prepare_pairs()
    fingerprints = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in CODE_SNAPSHOTS.items()}
    identity = {'config': current_run_config(), 'caption_origin': caption_origin,
        'selected_pairs': [{k: p[k] for k in ('index', 'sample_id', 'source_path', 'target_path')} for p in pairs],
        'implementation_sha256': fingerprints, 'source_notebook_sha256': V33_PARENT_NOTEBOOK_SHA256,
        'foa_revision': FOA_SOURCE_REVISION, 'kmeans_revision': KMEANS_REVISION}
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    resuming = RESUME_RUN_DIR is not None
    RUN_DIR = Path(RESUME_RUN_DIR) if resuming else OUTPUT_ROOT / f'kltn_attack_v3_3_{ATTENTION_MODE}_{utc_stamp()}'
    if resuming:
        previous = json.loads((RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
        if previous.get('resume_signature') != signature:
            raise ValueError('Resume configuration/data/code differs. Restore config or start a new run.')
        records_by_id = recover_completed_records(RUN_DIR, {pair['sample_id'] for pair in pairs})
        for record in records_by_id.values():
            validate_completed_artifacts(RUN_DIR, record, STEPS)
    else:
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        (RUN_DIR / 'images').mkdir()
        (RUN_DIR / 'captions').mkdir()
        config = {**identity, **current_run_config(), 'resume_signature': signature,
            'notebook': 'kltn-attack-v3-3.ipynb', 'logic_audit_version': 1, 'checkpoint_steps': checkpoint_steps(STEPS),
            'checkpoint_interval': max(1, STEPS // 10), 'dataset_root': str(DATASET_ROOT),
            'loss_history': 'reset per sample', 'seed_policy': 'PAPER_SEED + original source index',
            'guide_barrier': 'fresh checkpoint guide map before next update', 'saved_png_attention_input': True,
            'reference_provenance': REFERENCE_PROVENANCE, 'packages': PACKAGES}
        write_json(RUN_DIR / 'run_config.json', config)
        (RUN_DIR / 'code').mkdir()
        for name, source in CODE_SNAPSHOTS.items():
            (RUN_DIR / 'code' / f'{name}.py').write_text(source, encoding='utf-8')
        freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
        (RUN_DIR / 'requirements-resolved.txt').write_text(freeze.stdout, encoding='utf-8')
        records_by_id = {}
    manifest_path = RUN_DIR / 'manifest.jsonl'
    write_jsonl_atomic(manifest_path, records_by_id.values())
    status = {'status': 'running', 'phase': 'loading', 'samples': len(records_by_id),
              'requested_samples': len(pairs), 'errors_this_session': 0, 'resumed': resuming,
              'current_sample_id': None, 'sample_position': None, 'current_step': 0, 'steps_per_sample': STEPS}
    write_json(RUN_DIR / 'status.json', status)
    started = time.perf_counter()
    timings, active_sample, failures = [], None, []
    try:
        todo = [pair for pair in pairs if pair['sample_id'] not in records_by_id]
        captioner = None
        caption_metadata = None
        # Different models cannot coexist with Qwen attention on a 16GB T4. Build cached
        # captions first; then unload the caption model before loading the attention model.
        if todo and not reuse_attention_model_for_captioning():
            missing = [p for p in todo if not (RUN_DIR / 'captions' / f'{p["sample_id"]}.json').is_file()]
            previous_metadata = (json.loads((RUN_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
                if resuming and (RUN_DIR / 'model_metadata.json').is_file() else {})
            caption_revision = (previous_metadata.get('caption') or {}).get('resolved_revision')
            if not caption_revision:
                cached = sorted((RUN_DIR / 'captions').glob('*.json'))
                if cached:
                    first_cached = json.loads(cached[0].read_text(encoding='utf-8'))
                    caption_revision = first_cached.get('caption_model', {}).get('resolved_revision')
            status['phase'] = 'generating_captions'; write_json(RUN_DIR / 'status.json', status)
            external = VLMCaptioner(resolved_revision=caption_revision) if missing else None
            try:
                if external is not None:
                    caption_metadata = external.metadata
                elif previous_metadata.get('caption'):
                    caption_metadata = previous_metadata['caption']
                transform_for_captions = preprocess_image()
                for pending in tqdm(todo, desc='Auto captions (cache/recover)'):
                    source_cpu, target_cpu = read_preprocessed_pair(pending, transform_for_captions)
                    payloads, pils = _caption_images_from_tensors(source_cpu, target_cpu)
                    cached_caption = get_or_generate_caption_record(RUN_DIR, pending, external, payloads, pils)
                    expected_revision = (caption_metadata or {}).get('resolved_revision') or caption_revision
                    if expected_revision and cached_caption['caption_model'].get('resolved_revision') != expected_revision:
                        raise ValueError(f'Caption model revisions mixed in one run: {pending["sample_id"]}')
            finally:
                if external is not None:
                    external.close()
                export_caption_table(RUN_DIR)
        if todo:
            status['phase'] = 'loading_foa'; write_json(RUN_DIR / 'status.json', status)
            print(f'Loading FOA {PAPER_BACKBONES} on {ATTACK_DEVICE}; autocast={FOA_AUTOCAST}')
            extractor, models = ensure_foa_models()
            saved_metadata = (json.loads((RUN_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
                              if resuming and (RUN_DIR / 'model_metadata.json').is_file() else None)
            saved_revision = saved_metadata['attention']['resolved_revision'] if saved_metadata else None
            status['phase'] = 'loading_attention'; write_json(RUN_DIR / 'status.json', status)
            print(f'Loading {ATTENTION_MODEL_ID} on {ATTENTION_DEVICE}; revision={saved_revision or ATTENTION_REVISION}')
            backend = ensure_attention_backend(resolved_revision=saved_revision)
            if reuse_attention_model_for_captioning():
                captioner = VLMCaptioner(attention_backend=backend)
                caption_metadata = captioner.metadata
            if saved_metadata:
                old_caption = saved_metadata.get('caption')
                if old_caption and caption_metadata and old_caption.get('resolved_revision') != caption_metadata.get('resolved_revision'):
                    raise ValueError('Caption model revision differs from the resumed run.')
                if saved_revision != backend.metadata.get('resolved_revision'):
                    raise ValueError('Attention model revision differs from the saved run.')
                for saved_model, model in zip(saved_metadata.get('foa', []), models):
                    if saved_model.get('resolved_revision') != getattr(model.model.config, '_commit_hash', None):
                        raise ValueError('FOA model revision differs from the saved run; start a new run.')
            write_json(RUN_DIR / 'model_metadata.json', {'attention': backend.metadata, 'caption': caption_metadata,
                'foa': [{'backbone': name, 'model_id': getattr(model.model.config, '_name_or_path', None),
                         'resolved_revision': getattr(model.model.config, '_commit_hash', None),
                         'weights_dtype': 'float32', 'forward_autocast': FOA_AUTOCAST}
                        for name, model in zip(PAPER_BACKBONES, models)], 'torch': torch.__version__,
                'cuda': torch.version.cuda, 'gpus': {device: torch.cuda.get_device_name(torch.device(device))
                                                   for device in (ATTACK_DEVICE, ATTENTION_DEVICE)}})
        loading_seconds = time.perf_counter() - started
        status['phase'] = 'attack'; write_json(RUN_DIR / 'status.json', status)
        transform = preprocess_image()
        with tqdm(total=len(pairs) * STEPS, initial=len(records_by_id) * STEPS, desc=f'FOA {ATTENTION_MODE} (processed updates)', mininterval=2) as progress_bar:
            for position, pair in enumerate(pairs):
                stem = pair['sample_id']
                if stem in records_by_id:
                    continue
                sample_started = time.perf_counter()
                active_sample = RUN_DIR / 'samples' / stem
                if active_sample.exists():
                    # Preserve every failed attempt. Check resolved boundaries before moving.
                    backup = RUN_DIR / 'failed_samples' / f'{stem}_{utc_stamp()}'
                    if not active_sample.resolve().is_relative_to(RUN_DIR.resolve()) or not backup.resolve().is_relative_to(RUN_DIR.resolve()):
                        raise ValueError('Invalid sample artifact path.')
                    backup.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(active_sample), str(backup))
                active_sample.mkdir(parents=True, exist_ok=False)
                write_json(active_sample / 'status.json', {'status': 'running', 'sample_id': stem})
                observer = None
                progress_done = 0
                def report(phase, step, *, refresh=True):
                    status.update(phase=phase, current_sample_id=stem, sample_position=position + 1,
                                  current_step=step, samples=len(records_by_id), errors_this_session=len(failures),
                                  updated_at_utc=utc_stamp())
                    write_json(RUN_DIR / 'status.json', status)
                    progress_bar.set_postfix_str(
                        f'sample={position + 1}/{len(pairs)} id={stem} step={step}/{STEPS} '
                        f'ok={len(records_by_id)} errors={len(failures)} phase={phase}', refresh=refresh)
                report('preparing_sample', 0)
                try:
                    source_cpu, target_cpu = read_preprocessed_pair(pair, transform)
                    payloads, pils = _caption_images_from_tensors(source_cpu, target_cpu)
                    report('caption_generation', 0)
                    # Defaults reuse the *same* Qwen3-VL already loaded for attention.
                    # Alternative models were already captioned/cached in prepass.
                    caption_record = get_or_generate_caption_record(RUN_DIR, pair, captioner, payloads, pils)
                    guide, concepts = resolve_concepts(caption_record)
                    pair['caption_record'] = caption_record
                    pair['guide'], pair['concepts'] = guide, concepts
                    write_json(active_sample / 'sample_config.json', {**pair, 'seed': PAPER_SEED + pair['index']})
                    print(f'Sample {position + 1}/{len(pairs)} | id={stem} | generated guide={guide!r}')
                    # Captions are deterministic; reset attack seed AFTER caption generation.
                    set_foa_seed(PAPER_SEED + pair['index'])
                    loss_fn = FOAOptimalTransportLoss(models, PAPER_CLUSTER_NUMBER)
                    source = source_cpu.unsqueeze(0).to(ATTACK_DEVICE)
                    target = target_cpu.unsqueeze(0).to(ATTACK_DEVICE)
                    paths = {role: f'images/{stem}_{role}.png' for role in ('source', 'target', 'adversarial')}
                    (RUN_DIR / paths['source']).write_bytes(payloads[0])
                    (RUN_DIR / paths['target']).write_bytes(payloads[1])
                    observer = CheckpointObserver(active_sample, concepts, backend, live=LIVE_PREVIEW, total_steps=STEPS, epsilon=PAPER_EPSILON)
                    def progress(step):
                        nonlocal progress_done
                        report('attack', step, refresh=False)
                        progress_bar.update(step - progress_done)
                        progress_done = step
                    adversarial, trace = foa_attention_attack(source, target, extractor, loss_fn,
                        observer=observer, sample_dir=active_sample, mode=ATTENTION_MODE,
                        on_step=progress, on_phase=report, steps=STEPS, epsilon=PAPER_EPSILON, alpha=PAPER_ALPHA)
                    report('saving_artifacts', STEPS)
                    expected = set(checkpoint_steps(STEPS))
                    if observer.saved_steps != expected or observer.finished_steps != expected:
                        raise AssertionError('Checkpoint/map set is incomplete.')
                    shutil.copy2(active_sample / 'checkpoints' / f'step_{STEPS:05d}.png', RUN_DIR / paths['adversarial'])
                    with Image.open(RUN_DIR / paths['source']) as img:
                        original = np.asarray(img).astype(np.int16)
                    with Image.open(RUN_DIR / paths['adversarial']) as img:
                        saved = np.asarray(img).astype(np.int16)
                    linf = int(np.abs(saved - original).max())
                    if linf > math.ceil(PAPER_EPSILON):
                        raise AssertionError('Final PNG exceeds perturbation budget.')
                    figures = []
                    if PLOT_FIRST_N_SAMPLES is None or position < PLOT_FIRST_N_SAMPLES:
                        figures = plot_attention_timeline(active_sample, show=SHOW_FINAL_PLOTS)
                    record = {'index': pair['index'], 'sample_id': stem, 'original_path': pair['source_path'],
                        'target_path': pair['target_path'], 'linf_255': linf, **paths,
                        'image_sha256': {role: sha256_file(RUN_DIR / path) for role, path in paths.items()},
                        'attention_mode': ATTENTION_MODE, 'attention_dir': f'samples/{stem}/attention',
                        'guide_concept': guide, 'captions': caption_record['captions'],
                        'caption_file': f'captions/{stem}.json', 'checkpoints': checkpoint_steps(STEPS)}
                    timing = {'sample_id': stem, 'total_seconds': time.perf_counter() - sample_started,
                        'checkpoint_attention_seconds': observer.attention_seconds,
                        'guide_wait_seconds': observer.guide_wait_seconds, 'steps': STEPS,
                        'checkpoints': len(expected), 'seed': PAPER_SEED + pair['index']}
                    write_json(active_sample / 'timing.json', timing)
                    # Commit manifest before complete status, so interrupted commits are retried safely.
                    write_json(active_sample / 'manifest_record.json', record)
                    committed = {**records_by_id, stem: record}
                    write_jsonl_atomic(manifest_path, committed.values())
                    write_json(active_sample / 'status.json', {'status': 'complete', 'sample_id': stem,
                        'linf_255': linf, 'checkpoints': len(expected), 'figures': figures})
                    records_by_id[stem] = record
                    timings.append(timing)
                    report('sample_complete', STEPS)
                except Exception as error:
                    failure = {'sample_id': stem, 'status': 'error', 'error_type': type(error).__name__, 'error': str(error)}
                    failures.append(failure)
                    report('sample_error', progress_done)
                    write_json(active_sample / 'status.json', failure)
                    (active_sample / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
                    with (RUN_DIR / 'errors.jsonl').open('a', encoding='utf-8') as stream:
                        stream.write(json.dumps({**failure, 'time_utc': utc_stamp()}, ensure_ascii=False) + '\n')
                    if not CONTINUE_ON_ERROR:
                        raise
                    print(f'Sample {stem} failed: {type(error).__name__}: {error}')
                finally:
                    if observer is not None:
                        observer.close()
                    source = target = adversarial = trace = loss_fn = observer = None
                    source_cpu = target_cpu = payloads = pils = None
                    progress_bar.update(STEPS - progress_done)
                status.update(samples=len(records_by_id), errors_this_session=len(failures))
                write_json(RUN_DIR / 'status.json', status)
        status.update(status='complete' if len(records_by_id) == len(pairs) else 'partial', phase='complete',
                      current_sample_id=None, sample_position=None, current_step=None)
    except (Exception, KeyboardInterrupt) as error:
        status.update(status='interrupted' if isinstance(error, KeyboardInterrupt) else 'error', phase='stopped',
                      error_type=type(error).__name__, error=str(error))
        (RUN_DIR / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
        if active_sample is not None and active_sample.name not in records_by_id:
            write_json(active_sample / 'status.json', {**status, 'sample_id': active_sample.name})
        print(f'Run stopped: {type(error).__name__}: {error}. Partial artifacts are retained.')
    finally:
        status.update(samples=len(records_by_id), errors_this_session=len(failures))
        write_json(RUN_DIR / 'status.json', status)
        export_caption_table(RUN_DIR)
        total_seconds = time.perf_counter() - started
        write_json(RUN_DIR / 'summary.json', {'status': status['status'], 'completed_samples': len(records_by_id),
            'requested_samples': len(pairs), 'loading_seconds': locals().get('loading_seconds'),
            'session_seconds': total_seconds, 'per_sample_this_session': timings, 'failures_this_session': failures,
            'seconds_per_completed_sample_this_session': sum(t['total_seconds'] for t in timings) / len(timings) if timings else None})
    print('Output:', RUN_DIR, '| status:', status['status'])
    return RUN_DIR

remember_code('run')
if RUN_ATTACK:
    run_attention_experiment()
else:
    print('Definitions ready. RUN_ATTACK=True starts/resumes a run.')


## Xem lại một mẫu và đổi concept sau chạy

Cell inspect vẽ timeline từ maps có sẵn, không tải lại Qwen. Cell reprobe tính maps mới cho
tất cả PNG đã lưu, không chạy lại attack và không sửa checkpoint. Muốn mở run cũ trong session mới,
đặt `RUN_ATTACK=False`, chạy các cell định nghĩa, sửa đường dẫn ở cell cần dùng rồi chạy cell đó.
Chỉ reprobe: có thể dùng một GPU với `ATTENTION_DEVICE='cuda:0'`.

In [ ]:
RUN_INSPECT = False
INSPECT_RUN_DIR = None       # None: run hiện tại; hoặc Path('/kaggle/input/.../run_folder')
INSPECT_SAMPLE_ID = '00000'
if RUN_INSPECT:
    inspected_run = RUN_DIR if INSPECT_RUN_DIR is None else Path(INSPECT_RUN_DIR)
    if inspected_run is None:
        raise ValueError('Choose INSPECT_RUN_DIR.')
    plot_attention_timeline(inspected_run / 'samples' / INSPECT_SAMPLE_ID,
        figure_dir=OUTPUT_ROOT / 'inspections' / f'{INSPECT_SAMPLE_ID}_{utc_stamp()}', show=True)


In [ ]:
RUN_REPROBE = False
REPROBE_RUN_DIR = None      # vd Path('/kaggle/input/your-attack-run/kltn_v3_fixed_...')
REPROBE_SAMPLE_ID = '00000'    # None: mẫu đầu; hoặc '00000'
REPROBE_CONCEPTS = {'custom_0': 'a panda bear', 'custom_1': 'a dog with a toothbrush'}

def reprobe_saved_checkpoints(run_dir, sample_id, concepts, backend, analysis_root=None):
    run_dir = Path(run_dir)
    sample_dir = run_dir / 'samples' / sample_id
    files = sorted((sample_dir / 'checkpoints').glob('step_*.png'))
    if not files or int(files[0].stem.split('_')[-1]) != 0:
        raise ValueError('Saved run needs at least its step-0 PNG checkpoint.')
    if not concepts:
        raise ValueError('Provide at least one concept.')
    for label, text in concepts.items():
        if not isinstance(label, str) or not label or not all(c.isalnum() or c in '_-' for c in label):
            raise ValueError('Concept labels must be safe nonempty identifiers.')
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Concept text cannot be empty.')
    # Default writable destination, including when run_dir is read-only Kaggle Input.
    root = OUTPUT_ROOT / 'analyses' if analysis_root is None else Path(analysis_root)
    destination = root / f'{sample_id}_{utc_stamp()}'
    destination.mkdir(parents=True, exist_ok=False)
    attention_root = destination / 'attention'
    attention_root.mkdir()
    write_json(attention_root / 'concepts.json', concepts)
    image_hashes = {str(path): sha256_file(path) for path in files}
    status = {'status': 'running', 'checkpoints': 0}
    write_json(destination / 'status.json', status)
    write_json(destination / 'run_config.json', {'source_run': str(run_dir), 'sample_id': sample_id,
               'concepts': concepts, 'checkpoint_sha256': image_hashes,
               'attention_model': backend.metadata, 'prompt': ATTENTION_PROMPT,
               'content_words_only': CONTENT_WORDS_ONLY, 'max_concept_tokens': MAX_CONCEPT_TOKENS})
    initial_box = None
    initial_maps = {}
    try:
        original_metrics = sample_dir / 'attention' / 'checkpoint_metrics.jsonl'
        if original_metrics.is_file():
            first = next(line for line in original_metrics.read_text(encoding='utf-8').splitlines() if line.strip())
            initial_box = json.loads(first)['initial_hotspot_box']
        with preserve_rng(), (attention_root / 'checkpoint_metrics.jsonl').open('w', encoding='utf-8') as metric_stream:
            with Image.open(files[0]) as image:
                original = np.asarray(image.convert('RGB')).astype(np.int16)
            for path in files:
                step = int(path.stem.split('_')[-1])
                output_dir = attention_root / f'step_{step:05d}'; output_dir.mkdir()
                with Image.open(path) as image:
                    image = image.convert('RGB')
                    results, by_text = {}, {}
                    for label, text in concepts.items():
                        if text not in by_text:
                            by_text[text] = backend.extract(image, text)
                        result = by_text[text]; results[label] = result
                        np.savez_compressed(output_dir / f'{label}.npz', probability=result['probability'], minmax=result['minmax'])
                        write_json(output_dir / f'{label}.json', result['metadata'])
                    if initial_box is None:
                        first_result = next(iter(results.values()))
                        initial_box = build_schedule(first_result['minmax'], NUM_PHASES, HOTSPOT_TOP_K, HOTSPOT_IOU)[0]['box']
                    metrics = {}
                    for label, result in results.items():
                        probability = result['probability']
                        initial_maps.setdefault(label, probability.copy())
                        metrics[label] = {'initial_hotspot_mass': hotspot_grid_mass(probability, initial_box),
                            'l1_vs_initial': float(np.abs(probability - initial_maps[label]).sum()),
                            'mean_raw_vision_mass': result['metadata']['mean_raw_vision_mass']}
                    record = {'step': step, 'initial_hotspot_box': initial_box, 'concept_metrics': metrics,
                              'linf_saved_255': int(np.abs(np.asarray(image).astype(np.int16) - original).max())}
                    metric_stream.write(json.dumps(record, allow_nan=False) + '\n'); metric_stream.flush()
                    status['checkpoints'] += 1
                    write_json(destination / 'status.json', status)
        plot_attention_timeline(sample_dir, attention_root, destination / 'figures', show=True)
        status['status'] = 'complete'
    except (Exception, KeyboardInterrupt) as error:
        status.update(status='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                      error_type=type(error).__name__, error=str(error))
        (destination / 'traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
        raise
    finally:
        unchanged = all(sha256_file(path) == digest for path, digest in image_hashes.items())
        status['original_checkpoints_unchanged'] = unchanged
        write_json(destination / 'status.json', status)
        if not unchanged:
            raise AssertionError('Original checkpoint bytes changed during reprobe.')
    archive = pack_artifacts(destination)
    print('Reprobe output:', destination); display(FileLink(str(archive)))
    return destination

remember_code('reprobe')
if RUN_REPROBE:
    selected_run = RUN_DIR if REPROBE_RUN_DIR is None else Path(REPROBE_RUN_DIR)
    if selected_run is None:
        raise ValueError('Set REPROBE_RUN_DIR or run an attack first.')
    candidates = sorted(path.name for path in (selected_run / 'samples').iterdir() if path.is_dir())
    if not candidates:
        raise ValueError('No sample directories in selected run.')
    selected_id = candidates[0] if REPROBE_SAMPLE_ID is None else REPROBE_SAMPLE_ID
    reprobe_saved_checkpoints(selected_run, selected_id, REPROBE_CONCEPTS, ensure_attention_backend())
else:
    print('Reprobe is off. Set RUN_REPROBE=True to analyze saved checkpoints with new text.')


In [ ]:
# Can run separately after interruption/failure to export partial artifacts.
if RUN_DIR is not None and RUN_DIR.is_dir():
    archive = pack_artifacts(RUN_DIR)
    print('Run status:', json.loads((RUN_DIR / 'status.json').read_text(encoding='utf-8')))
    for path in (RUN_DIR / 'manifest.jsonl', RUN_DIR / 'run_config.json', RUN_DIR / 'summary.json', archive):
        if path.is_file():
            display(FileLink(str(path)))
else:
    print('No attack run to export in this session.')
